<a href="https://colab.research.google.com/github/lmaas37/FDD-Week-1/blob/main/05_cx_ensembles_student.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Ensemble Methods for Predictive Maintenance

## The problem

Industrial machines such as motors, pumps and compressors are monitored in real time by
sensors that record temperature, vibration, pressure, rotation speed, current draw, oil
level and more. When a machine is heading towards a breakdown, these readings drift into
abnormal patterns well before it actually stops.

**Predictive maintenance** is the task of using those sensor readings to predict whether a
machine is at risk of **failure**, so that it can be serviced *before* it breaks. An
unplanned breakdown halts production, can damage equipment, and is far more expensive than
planned maintenance. It is therefore a risk management problem: we want to catch as many
genuine failures as possible while keeping false alarms low.

In this notebook every row is one machine, described by its current sensor readings plus a
few categorical descriptors (machine type, work shift, plant). The target column `failure`
is 1 if the machine fails and 0 otherwise, and the two classes are balanced.

## What we will do

We compare a sequence of increasingly powerful models and measure how much each idea adds:

1. A single **Decision Tree**, our baseline.
2. A **Random Forest**, plus how to read feature importance and select features.
3. Gradient boosting with **XGBoost** and **LightGBM**.
4. A **Stacking** ensemble that blends several models with a meta learner.

Every model is scored with **ROC AUC** (how well it ranks machines by risk) and
**accuracy** (meaningful here because the classes are balanced).

In [1]:
#@title 🗺️ Roadmap: what we'll see in this notebook { display-mode: "form" }
from IPython.display import HTML, display

def _roadmap():
    steps = [("🎯", "Variance", "why one model wobbles"),
             ("🌳", "Decision Tree", "the baseline — it overfits"),
             ("🌲", "Random Forest", "bagging cuts variance"),
             ("🚀", "Boosting", "XGBoost & LightGBM cut bias"),
             ("🧩", "Stacking", "blend complementary models"),
             ("🏆", "Compare", "rank every model")]
    grad = ["#667eea", "#7b71ee", "#9a6fe2", "#b06fd0", "#c56fbe", "#db6fa9"]
    blocks = ""
    for (ic, t, d), g in zip(steps, grad):
        blocks += (f'<div class="rm-step"><div class="rm-ic" style="background:linear-gradient(135deg,{g},{g}cc)">{ic}</div>'
                   f'<div class="rm-t">{t}</div><div class="rm-d">{d}</div></div>'
                   '<div class="rm-ar">➜</div>')
    blocks = blocks.rsplit('<div class="rm-ar">➜</div>', 1)[0]
    display(HTML(f'''
<style>
.rm{{font-family:system-ui,Segoe UI,Roboto,sans-serif;background:linear-gradient(135deg,#f6f8ff,#fbf5ff);
    border-radius:18px;padding:20px 16px;margin:8px 0;border:1px solid #ecebff}}
.rm-h{{font-size:20px;font-weight:800;color:#3b2d6b;margin:0 0 4px}}
.rm-s{{font-size:12px;color:#6b6685;margin:0 0 16px}}
.rm-row{{display:flex;align-items:flex-start;flex-wrap:wrap;gap:0}}
.rm-step{{flex:1 1 104px;min-width:104px;text-align:center;padding:0 2px}}
.rm-ic{{width:50px;height:50px;border-radius:50%;margin:0 auto 8px;display:flex;align-items:center;
       justify-content:center;font-size:22px;color:#fff;box-shadow:0 6px 14px rgba(102,126,234,.35)}}
.rm-t{{font-weight:800;font-size:12.5px;color:#2c2350}}
.rm-d{{font-size:10px;color:#8b86a6;margin-top:3px;line-height:1.3}}
.rm-ar{{display:flex;align-items:center;font-size:18px;color:#b9a9e6;flex:0 0 14px;height:50px}}
</style>
<div class="rm">
  <div class="rm-h">🗺️ What we'll build in this notebook</div>
  <div class="rm-s">Each step is one ensemble idea, getting stronger as we go &mdash; from a single shaky tree to a blend of complementary models.</div>
  <div class="rm-row">{blocks}</div>
</div>'''))

_roadmap()


## Reminder: what is variance?

In statistics, **variance** measures how spread out values are around their average. In
machine learning, the variance of a model is how much its predictions would change if we
retrained it on a different sample of data. A high variance model is unstable: its
predictions scatter a lot depending on the exact training set.

**High variance causes overfitting**: a model that swings a lot with the training data ends
up chasing noise and generalises poorly to new machines. Averaging many trees, as a Random
Forest does, lowers the variance and so reduces overfitting.

In [2]:
#@title 📊 Visualization: variance = how much a model moves when you resample (double-click to view the code) { display-mode: "form" }
from IPython.display import HTML, display
import uuid as _uuid

_uid = "vz" + _uuid.uuid4().hex[:8]

_HTML = r"""
<style>
#__UID__{font-family:system-ui,Segoe UI,Roboto,sans-serif;background:linear-gradient(135deg,#f6f8ff,#fbf5ff);
  border:1px solid #ecebff;border-radius:18px;padding:18px;max-width:860px;color:#2c2350}
#__UID__ .vz-h{font-size:19px;font-weight:800;color:#3b2d6b;margin:0 0 3px}
#__UID__ .vz-s{font-size:12px;color:#6b6685;margin:0 0 13px;line-height:1.5}
#__UID__ .vz-s b{color:#3b2d6b}
#__UID__ .vz-row{display:flex;gap:14px;flex-wrap:wrap;align-items:stretch}
#__UID__ .vz-plot{flex:1 1 360px;background:#fff;border:1px solid #e6e8ee;border-radius:14px;padding:10px}
#__UID__ .vz-side{flex:1 1 170px;background:#fff;border:1px solid #e6e8ee;border-radius:14px;padding:11px 13px;display:flex;flex-direction:column}
#__UID__ .vz-st{font-size:11.5px;font-weight:800;color:#2c2350;margin-bottom:2px}
#__UID__ .vz-ss{font-size:10px;color:#8b86a6;margin-bottom:6px;line-height:1.3}
#__UID__ .vz-spread{font-size:11px;margin-top:8px;line-height:1.7}
#__UID__ .vz-spread b{font-weight:800}
#__UID__ .vz-ctrl{display:flex;gap:8px;align-items:center;margin-top:13px;flex-wrap:wrap}
#__UID__ .vz-btn{cursor:pointer;border:none;border-radius:10px;padding:10px 14px;font-size:13px;font-weight:700;color:#fff;
  background:linear-gradient(135deg,#667eea,#764ba2);box-shadow:0 6px 14px rgba(102,126,234,.32)}
#__UID__ .vz-btn.alt{background:linear-gradient(135deg,#8b86a6,#6b6685);box-shadow:none}
#__UID__ .vz-btn:active{transform:translateY(1px)}
#__UID__ .vz-leg{font-size:10.5px;color:#8b86a6;margin-top:7px;display:flex;gap:13px;flex-wrap:wrap}
#__UID__ .vz-sw{display:inline-block;width:13px;height:3px;border-radius:2px;vertical-align:middle;margin-right:5px}
#__UID__ .vz-cap{font-size:11.5px;color:#6b6685;line-height:1.5;margin-top:12px;background:#fff;border:1px solid #ecebff;
  border-radius:11px;padding:9px 11px;min-height:18px}#__UID__ .vz-cap b{color:#3b2d6b}
</style>
<div id="__UID__">
  <div class="vz-h">🎯 What "variance" means</div>
  <div class="vz-s">Same task each time: fit a model to a fresh random sample of noisy data. A
  <b style="color:#e0796d">single deep tree</b> chases the noise, so its curve <b>lurches</b> every time the data
  changes &mdash; that instability <i>is</i> variance. The <b style="color:#764ba2">average of many trees</b> stays put.
  Press <b>Resample</b> a few times and watch the ✕ point.</div>

  <div class="vz-row">
    <div class="vz-plot">
      <svg class="vz-svg" viewBox="0 0 380 210" width="100%" style="overflow:visible"></svg>
      <div class="vz-leg">
        <span><span class="vz-sw" style="background:#b8b4c8"></span>true pattern</span>
        <span><span class="vz-sw" style="background:#e0796d"></span>one deep tree</span>
        <span><span class="vz-sw" style="background:#764ba2"></span>average of many</span>
      </div>
    </div>
    <div class="vz-side">
      <div class="vz-st">Prediction at the ✕ point</div>
      <div class="vz-ss">each resample drops one dot &mdash; spread = variance</div>
      <svg class="vz-track" viewBox="0 0 150 150" width="100%" style="overflow:visible"></svg>
      <div class="vz-spread"></div>
    </div>
  </div>

  <div class="vz-ctrl">
    <button class="vz-btn" data-act="resample">🔄 Resample the data</button>
    <button class="vz-btn" data-act="auto">⏩ Auto</button>
    <button class="vz-btn alt" data-act="reset">↺</button>
  </div>
  <div class="vz-cap">Press <b>🔄 Resample the data</b> to draw a new sample and refit both models.</div>
</div>
<script>
(function(){
  var root=document.getElementById("__UID__");
  var svg=root.querySelector(".vz-svg"), track=root.querySelector(".vz-track");
  var spread=root.querySelector(".vz-spread"), cap=root.querySelector(".vz-cap");
  var bR=root.querySelector('[data-act="resample"]'), bA=root.querySelector('[data-act="auto"]'), bT=root.querySelector('[data-act="reset"]');
  var SVGNS="http://www.w3.org/2000/svg";
  var W=380,H=210,P=16,pw=W-2*P,ph=H-2*P, G=64, N=22, K=9, B=24, XSTAR=0.5;
  var seed=12345, singles=[], ens=[], timer=null;

  function rnd(){seed=(seed*1664525+1013904223)>>>0;return seed/4294967296;}
  function gauss(){return (rnd()+rnd()+rnd()+rnd()-2)/1.2;}
  function truth(x){return 0.5+0.32*Math.sin(2*Math.PI*1.05*x - 0.4);}
  function sample(){var xs=[],ys=[];for(var i=0;i<N;i++){var x=rnd();xs.push(x);ys.push(truth(x)+0.14*gauss());}return {xs:xs,ys:ys};}
  function fitTree(s){                              // piecewise-constant "deep-ish" tree over K bins
    var sum=new Array(K).fill(0), cnt=new Array(K).fill(0);
    for(var i=0;i<N;i++){var k=Math.max(0,Math.min(K-1,Math.floor(s.xs[i]*K)));sum[k]+=s.ys[i];cnt[k]++;}
    var m=new Array(K); var g=s.ys.reduce(function(a,b){return a+b;},0)/N;
    for(var k=0;k<K;k++) m[k]=cnt[k]?sum[k]/cnt[k]:null;
    for(var k=0;k<K;k++){if(m[k]===null){var d=1;while(d<K){if(k-d>=0&&m[k-d]!==null){m[k]=m[k-d];break;}if(k+d<K&&cnt[k+d]){m[k]=sum[k+d]/cnt[k+d];break;}d++;}if(m[k]===null)m[k]=g;}}
    return m;
  }
  function predOf(m,x){return m[Math.max(0,Math.min(K-1,Math.floor(x*K)))];}

  var curSample=null, curTree=null, curEns=null;   // arrays over grid
  function build(){
    curSample=sample(); curTree=fitTree(curSample);
    var acc=new Array(G).fill(0);
    for(var b=0;b<B;b++){var mb=fitTree(sample());for(var i=0;i<G;i++)acc[i]+=predOf(mb,i/(G-1));}
    curEns=acc.map(function(v){return v/B;});
    singles.push(predOf(curTree,XSTAR));
    ens.push(curEns[Math.round(XSTAR*(G-1))]);
    if(singles.length>16){singles.shift();ens.shift();}
  }
  function el(n,a){var e=document.createElementNS(SVGNS,n);for(var k in a)e.setAttribute(k,a[k]);return e;}
  function X(x){return P+x*pw;} function Y(v){return P+(1-v)*ph;}

  function drawMain(){
    svg.innerHTML="";
    svg.appendChild(el("rect",{x:P,y:P,width:pw,height:ph,fill:"#fbfcff",stroke:"#eef0f5",rx:8}));
    // true curve
    var tp=[];for(var i=0;i<G;i++)tp.push(X(i/(G-1)).toFixed(1)+","+Y(truth(i/(G-1))).toFixed(1));
    svg.appendChild(el("polyline",{points:tp.join(" "),fill:"none",stroke:"#b8b4c8","stroke-width":2,"stroke-dasharray":"5 4"}));
    // sample points
    for(var i=0;i<N;i++) svg.appendChild(el("circle",{cx:X(curSample.xs[i]).toFixed(1),cy:Y(curSample.ys[i]).toFixed(1),r:2.4,fill:"#cfd2dd"}));
    // single tree (jagged)
    var sp=[];for(var i=0;i<G;i++)sp.push(X(i/(G-1)).toFixed(1)+","+Y(predOf(curTree,i/(G-1))).toFixed(1));
    svg.appendChild(el("polyline",{points:sp.join(" "),fill:"none",stroke:"#e0796d","stroke-width":1.9,"stroke-linejoin":"round"}));
    // ensemble (smooth)
    var ep=[];for(var i=0;i<G;i++)ep.push(X(i/(G-1)).toFixed(1)+","+Y(curEns[i]).toFixed(1));
    svg.appendChild(el("polyline",{points:ep.join(" "),fill:"none",stroke:"#764ba2","stroke-width":2.6,"stroke-linejoin":"round"}));
    // marked x*
    svg.appendChild(el("line",{x1:X(XSTAR),y1:P,x2:X(XSTAR),y2:P+ph,stroke:"#3b2d6b","stroke-width":1,"stroke-dasharray":"2 3",opacity:.5}));
    svg.appendChild(el("circle",{cx:X(XSTAR),cy:Y(predOf(curTree,XSTAR)),r:4,fill:"#e0796d",stroke:"#fff","stroke-width":1.5}));
    svg.appendChild(el("circle",{cx:X(XSTAR),cy:Y(curEns[Math.round(XSTAR*(G-1))]),r:4,fill:"#764ba2",stroke:"#fff","stroke-width":1.5}));
    var tx=el("text",{x:X(XSTAR),y:P+ph+12,"text-anchor":"middle","font-size":10,fill:"#3b2d6b","font-weight":"700"});
    tx.appendChild(document.createTextNode("✕")); svg.appendChild(tx);
  }
  function std(a){if(a.length<2)return 0;var m=a.reduce(function(p,q){return p+q;},0)/a.length;return Math.sqrt(a.reduce(function(p,q){return p+(q-m)*(q-m);},0)/a.length);}
  function drawTrack(){
    track.innerHTML="";
    var w=150,h=150,pp=12, colS=48, colE=110;
    track.appendChild(el("line",{x1:pp,y1:h-pp,x2:w-pp,y2:h-pp,stroke:"#eef0f5","stroke-width":1}));
    [["single",colS,"#e0796d"],["average",colE,"#764ba2"]].forEach(function(c){
      var t=el("text",{x:c[1],y:h-2,"text-anchor":"middle","font-size":9,fill:c[2],"font-weight":"700"});t.appendChild(document.createTextNode(c[0]));track.appendChild(t);
    });
    function Yt(v){return pp+(1-v)*(h-2*pp-8);}
    singles.forEach(function(v,i){var last=i===singles.length-1;track.appendChild(el("circle",{cx:colS+(i%5-2)*3,cy:Yt(v).toFixed(1),r:last?4:3,fill:"#e0796d",opacity:last?1:.4}));});
    ens.forEach(function(v,i){var last=i===ens.length-1;track.appendChild(el("circle",{cx:colE+(i%5-2)*3,cy:Yt(v).toFixed(1),r:last?4:3,fill:"#764ba2",opacity:last?1:.4}));});
    spread.innerHTML='<span style="color:#e0796d">single tree</span> spread: <b style="color:#e0796d">±'+std(singles).toFixed(2)+
      '</b><br><span style="color:#764ba2">average</span> spread: <b style="color:#764ba2">±'+std(ens).toFixed(2)+'</b>';
  }
  function narrate(){
    var n=singles.length;
    if(n<2){cap.innerHTML='One sample in. Resample again &mdash; watch the <b style="color:#e0796d">red</b> ✕ dot jump while the <b style="color:#764ba2">purple</b> one stays put.';return;}
    cap.innerHTML='After <b>'+n+'</b> resamples the single tree\'s prediction is scattered (<b style="color:#e0796d">±'+std(singles).toFixed(2)+
      '</b>), while the average barely budges (<b style="color:#764ba2">±'+std(ens).toFixed(2)+'</b>). <b>That gap is variance &mdash; and averaging removes most of it.</b>';
  }
  function step(){build();drawMain();drawTrack();narrate();}
  function stopAuto(){if(timer){clearInterval(timer);timer=null;bA.textContent="⏩ Auto";}}
  function toggleAuto(){if(timer){stopAuto();return;}bA.textContent="⏸";timer=setInterval(step,950);}
  function reset(){stopAuto();singles=[];ens=[];seed=(Date.now()>>>0)||777;step();
    cap.innerHTML='Press <b>🔄 Resample the data</b> to draw a new sample and refit both models.';}
  bR.addEventListener("click",step); bA.addEventListener("click",toggleAuto); bT.addEventListener("click",reset);
  reset();
})();
</script>
""".replace("__UID__", _uid)

display(HTML(_HTML))


In [4]:
#@title Setup { display-mode: "form" }
import warnings; warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_predict, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LinearRegression
from sklearn.metrics import roc_auc_score, accuracy_score, confusion_matrix
from xgboost import XGBClassifier
try:
    from lightgbm import LGBMClassifier
except Exception:
    import subprocess, sys
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'lightgbm'])
    from lightgbm import LGBMClassifier
RANDOM_STATE = 0

In [5]:
#@title Load the machine sensor dataset { display-mode: "form" }
def _build_dataset(n=4000, seed=42, balance=0.5, n_noise=35):
    rng = np.random.default_rng(seed)
    mode = rng.integers(0, 4, n)
    c_t = np.array([55., 72., 63., 82.]); c_v = np.array([2., 5., 4., 3.2])
    temperature = rng.normal(c_t[mode], 6., n)
    vibration = np.clip(rng.normal(c_v[mode], 0.8, n), 0, None)
    pressure = rng.normal(30., 5., n)
    rotation_speed = rng.normal(1500., 200., n)
    motor_current = rng.normal(10., 2., n) + 0.02 * (temperature - 60)
    oil_level = np.clip(rng.normal(70., 15., n), 0, 100)
    acoustic_db = rng.normal(60., 8., n) + 1.5 * vibration
    machine_type = rng.choice(['A','B','C'], size=n, p=[0.4,0.35,0.25])
    shift = rng.choice(['day','night','swing'], size=n)
    plant = rng.choice(['north','south'], size=n)
    type_risk = pd.Series(machine_type).map({'A':-0.4,'B':0.0,'C':0.6}).to_numpy()
    def z(a): a = a.astype(float); return (a - a.mean()) / (a.std() + 1e-9)
    s = (1.2*z(temperature) + 0.8*z(vibration) + 0.5*z(motor_current)
         - 0.4*z(oil_level) + 0.3*z(rotation_speed) + type_risk)
    s += 0.7*2.0*((temperature>75)&(vibration>4.0))
    s += 0.7*2.6*((temperature>78)&(vibration>4.5)&(pressure<28))
    s += 0.7*2.4*((motor_current>13)&(rotation_speed>1650)&(oil_level<55))
    s -= 0.7*2.6*((oil_level>85)&(temperature<60)&(vibration<3)&(motor_current<9))
    u = (z(temperature)+z(vibration))/np.sqrt(2.0); w = (z(pressure)-z(rotation_speed))/np.sqrt(2.0)
    s += 1.5*4.6*np.sin(0.9*u)*np.cos(0.9*w)
    s += 1.5*3.4*np.sin(0.8*w)*np.cos(0.8*u)
    pa,pb,pc,pd_ = (rng.normal(0,1,n) for _ in range(4))
    s += 1.1*2.6*np.where((pa>0)==(pb>0),1.0,-1.0)
    s += 1.1*2.2*np.where((pc>0)==(pd_>0),1.0,-1.0)
    s = s + rng.normal(0, 3.0, n)
    failure = (s > np.quantile(s, 1.0-balance)).astype(int)
    df = pd.DataFrame({'temperature':temperature.round(2),'vibration':vibration.round(3),
        'pressure':pressure.round(2),'rotation_speed':rotation_speed.round(1),
        'motor_current':motor_current.round(3),'oil_level':oil_level.round(1),
        'acoustic_db':acoustic_db.round(2),'phase_a':pa.round(3),'phase_b':pb.round(3),
        'phase_c':pc.round(3),'phase_d':pd_.round(3),'machine_type':machine_type,
        'shift':shift,'plant':plant})
    for i in range(1, n_noise+1): df[f'sensor_noise_{i}'] = rng.normal(0,1,n).round(3)
    oi = rng.choice(n, size=max(1, n//400), replace=False)
    df.loc[oi,'temperature'] *= rng.uniform(1.3,1.8,size=oi.size)
    df['failure'] = failure
    return df

df = _build_dataset()
print('rows x columns:', df.shape, '   failure rate: {:.0%}'.format(df['failure'].mean()))
df.head()

rows x columns: (4000, 50)    failure rate: 50%


,temperature,vibration,pressure,rotation_speed,motor_current,oil_level,acoustic_db,phase_a,phase_b,phase_c,...,sensor_noise_27,sensor_noise_28,sensor_noise_29,sensor_noise_30,sensor_noise_31,sensor_noise_32,sensor_noise_33,sensor_noise_34,sensor_noise_35,failure
0,48.85,2.104,38.67,1501.9,13.588,100.0,63.47,-0.371,-0.462,0.230,...,1.087,0.644,1.410,-0.083,-1.276,-0.198,-1.287,-0.918,-0.118,1
1,87.45,2.471,33.18,1452.0,15.265,77.2,66.44,2.326,0.586,1.853,...,-0.174,1.723,2.440,-0.200,1.332,-0.377,0.280,0.123,0.632,1
2,55.78,4.624,42.31,1327.7,10.566,78.1,75.68,-0.663,0.519,0.459,...,-0.397,-2.296,-0.534,-0.478,0.661,0.321,-0.363,2.121,-0.280,1
3,70.28,3.841,26.09,1407.2,10.662,67.3,74.60,0.745,-1.689,0.803,...,-0.027,0.629,-1.316,-1.081,0.809,-0.264,-0.447,-0.831,0.255,0
4,68.48,5.993,21.18,1464.4,10.759,87.4,54.86,1.797,1.466,0.141,...,0.524,0.073,0.587,0.387,-0.368,1.046,-0.822,1.746,-0.078,1


In [6]:
#@title Encode the categorical columns { display-mode: "form" }
df_enc = pd.get_dummies(df, columns=['machine_type','shift','plant'], drop_first=True)
y = df_enc['failure'].astype(int)
X = df_enc.drop(columns='failure')

## Train and test split

We always judge a model on data it has never seen during training. We hold out 25 percent
of the machines as a test set, and we stratify the split so the failure rate is the same
in the training and test parts.

scikit-learn reference:
[train_test_split](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html).

In [8]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify=y, random_state=RANDOM_STATE)


print('training machines:', X_train.shape[0])
print('test machines    :', X_test.shape[0])
print('failure rate  train: {:.0%}   test: {:.0%}'.format(y_train.mean(), y_test.mean()))

training machines: 3000
test machines    : 1000
failure rate  train: 50%   test: 50%


In [ ]:
#@title Extra preparation used only by the SVM { display-mode: "form" }
scaler = StandardScaler().fit(X_train)
X_train_sc, X_test_sc = scaler.transform(X_train), scaler.transform(X_test)
selector = SelectKBest(f_classif, k=10).fit(X_train_sc, y_train)
X_train_svm, X_test_svm = selector.transform(X_train_sc), selector.transform(X_test_sc)

In [ ]:
#@title 📊 How we measure a model: Accuracy vs ROC AUC (double-click to view the code) { display-mode: "form" }
from IPython.display import HTML, display

display(HTML(r'''
<style>
.mm{font-family:system-ui,Segoe UI,Roboto,sans-serif;background:linear-gradient(135deg,#f6f8ff,#fbf5ff);
    border:1px solid #ecebff;border-radius:18px;padding:18px;max-width:840px;color:#2c2350}
.mm-h{font-size:20px;font-weight:800;color:#3b2d6b;margin:0 0 3px}
.mm-s{font-size:12px;color:#6b6685;margin:0 0 14px;line-height:1.45}
.mm-row{display:flex;gap:13px;flex-wrap:wrap}
.mm-card{flex:1 1 340px;background:#fff;border:1px solid #e6e8ee;border-radius:14px;padding:14px 16px}
.mm-tag{display:inline-block;font-size:10px;font-weight:800;border-radius:7px;padding:3px 9px;margin-bottom:8px}
.mm-ct{font-size:15px;font-weight:800;color:#2c2350;margin-bottom:4px}
.mm-formula{font-family:ui-monospace,Menlo,Consolas,monospace;font-size:11.5px;color:#3b2d6b;background:#f3eefb;
    border-radius:8px;padding:7px 10px;margin:7px 0;line-height:1.5}
.mm-desc{font-size:11.5px;color:#6b6685;line-height:1.5}.mm-desc b{color:#3b2d6b}
.mm-scale{display:flex;align-items:center;gap:8px;margin-top:11px;font-size:10px;color:#8b86a6;font-weight:700}
.mm-bar{flex:1;height:9px;border-radius:6px;background:linear-gradient(90deg,#e0796d,#e0a23c,#39b36a)}
.mm-rank{display:flex;align-items:flex-end;gap:5px;height:74px;margin-top:10px;padding:8px;background:#fbfbff;
    border:1px solid #f0f0f6;border-radius:10px}
.mm-dot{flex:1;display:flex;flex-direction:column;align-items:center;justify-content:flex-end;height:100%}
.mm-circ{width:15px;height:15px;border-radius:50%;border:2px solid #fff;box-shadow:0 1px 3px rgba(0,0,0,.15)}
.mm-thr{font-size:9px;color:#a7a3bd;margin-top:6px;text-align:center}
.mm-cap{font-size:11.5px;color:#6b6685;line-height:1.5;margin-top:13px;background:#fff;border:1px solid #ecebff;
    border-radius:11px;padding:9px 11px}.mm-cap b{color:#3b2d6b}
</style>
<div class="mm">
  <div class="mm-h">📏 How we measure a model</div>
  <div class="mm-s">Every model outputs a <b>risk score</b> per machine. Two different questions about that score:</div>
  <div class="mm-row">

    <div class="mm-card">
      <div class="mm-tag" style="background:#eef3ff;color:#4d6fd6">✔️ ACCURACY</div>
      <div class="mm-ct">Did we label it right?</div>
      <div class="mm-formula">accuracy = correct predictions / all machines</div>
      <div class="mm-desc">Pick a cutoff (usually 0.5), call everything above it &ldquo;fail,&rdquo; and count the share you got
      right. Simple &mdash; but it <b>depends entirely on the cutoff</b> you chose.</div>
      <div class="mm-scale"><span>0.5 guessing</span><div class="mm-bar"></div><span>1.0 perfect</span></div>
    </div>

    <div class="mm-card">
      <div class="mm-tag" style="background:#f3eefb;color:#8a4fd0">📈 ROC AUC &nbsp;<span style="font-weight:600">(our main metric)</span></div>
      <div class="mm-ct">Did we rank it right?</div>
      <div class="mm-formula">AUC = P( risk(failing) &gt; risk(healthy) )</div>
      <div class="mm-desc">The chance that a random <b>failing</b> machine gets a higher risk than a random <b>healthy</b>
      one. It judges the <b>ordering</b> across <i>every</i> threshold at once &mdash; so it never depends on where we put
      the cutoff.</div>
      <div class="mm-rank">
        <div class="mm-dot" style="height:38%"><div class="mm-circ" style="background:#39b36a"></div></div>
        <div class="mm-dot" style="height:30%"><div class="mm-circ" style="background:#39b36a"></div></div>
        <div class="mm-dot" style="height:62%"><div class="mm-circ" style="background:#e0796d"></div></div>
        <div class="mm-dot" style="height:50%"><div class="mm-circ" style="background:#39b36a"></div></div>
        <div class="mm-dot" style="height:78%"><div class="mm-circ" style="background:#e0796d"></div></div>
        <div class="mm-dot" style="height:90%"><div class="mm-circ" style="background:#e0796d"></div></div>
      </div>
      <div class="mm-thr">🔴 failing machines ranked higher than 🟢 healthy ones = high AUC</div>
    </div>

  </div>
  <div class="mm-cap">We report both, but lean on <b>ROC AUC</b>: with a risk model you care about <b>who is most likely to
  fail</b> (the ranking), not whether a fixed 0.5 line happens to be in the right place. On these balanced classes,
  <b>0.5 = coin-flip</b> and <b>1.0 = perfect</b> for both metrics. <span style="color:#a7a3bd">(sklearn:
  accuracy_score, roc_auc_score)</span></div>
</div>
'''))

We collect every model's test ROC AUC in a dictionary so we can compare them at the end.

In [ ]:
results = {}   # model name -> ROC AUC on the test set

## Exercise 1. Decision Tree

A decision tree splits the data with simple rules on one feature at a time. It is easy to
read, but a single tree has high variance and limited capacity: it can only express a few
rules at once. This is our baseline, and every ensemble that follows should beat it.

**Task.** Train a `DecisionTreeClassifier` with `max_depth=3` and evaluate it.

scikit-learn reference:
[DecisionTreeClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeClassifier.html),
[decision trees guide](https://scikit-learn.org/stable/modules/tree.html).

### Your turn

In [ ]:
# 🎯 YOUR TURN — Exercise 1: the baseline, a single decision tree.
#
# 💭 Think first: we cap the tree at depth 3 on purpose. A much deeper tree can keep splitting until
#    it classifies every TRAINING machine perfectly — so why might that hurt it on machines it has
#    never seen? (You will test your guess in the depth plot a few cells below.)
#
# 🎯 Implement: create a decision tree limited to max_depth=3 (use random_state=RANDOM_STATE) and
#    fit it on X_train, y_train. Store it in `dt`.
dt = ...

# 🎯 YOUR TURN — score the model by filling in the four lines below.
# 💭 The tools you need (work out the right model and data yourself):
#      • model.predict_proba(X)[:, 1]     -> predicted probability of FAILURE (column 1)
#      • (proba >= 0.5).astype(int)       -> turn those probabilities into 0/1 labels at the 0.5 cutoff
#      • roc_auc_score(y_true, proba)     -> ROC AUC  (takes the PROBABILITIES)
#      • accuracy_score(y_true, y_pred)   -> accuracy (takes the 0/1 PREDICTIONS)
proba = ...   # failure probabilities on the test set
pred  = ...   # 0/1 predictions from proba at threshold 0.5
auc   = ...   # ROC AUC from y_test and proba
acc   = ...   # accuracy from y_test and pred

# already written for you: store and print
results['DecisionTree'] = auc
print(f'Decision Tree   ROC-AUC = {auc:.3f}   Accuracy = {acc:.3f}')

The **confusion matrix** shows where the errors are. The bottom left cell counts the
**missed failures** (machines that broke but we predicted ok), the most expensive mistake
in maintenance.

In [ ]:
#@title Confusion matrix { display-mode: "form" }
cm = confusion_matrix(y_test, pred)
fig, ax = plt.subplots(figsize=(3.2,3)); ax.imshow(cm, cmap='Blues')
for (i,j),v in np.ndenumerate(cm):
    ax.text(j,i,str(v),ha='center',va='center',color='white' if v>cm.max()/2 else 'black')
ax.set_xticks([0,1]); ax.set_yticks([0,1])
ax.set_xticklabels(['ok','fail']); ax.set_yticklabels(['ok','fail'])
ax.set_xlabel('predicted'); ax.set_ylabel('actual'); ax.set_title('Decision Tree')
plt.tight_layout(); plt.show()

The picture below shows the tree. Each box is a decision: it tests one sensor against a
threshold (for example `temperature <= 75`), and a machine goes left if the test is true
and right otherwise. The colour shows the predicted class (one colour for `ok`, the other
for `fail`) and the deeper the colour the purer that group is. Reading a path from the top
down to a leaf gives the rule the tree uses to classify a machine. We only draw the first
two levels to keep it readable.

In [ ]:
#@title Visualise the tree { display-mode: "form" }
plt.figure(figsize=(16,5))
plot_tree(dt, feature_names=list(X_train.columns), max_depth=2, filled=True,
          fontsize=8, class_names=['ok','fail'])
plt.show()

### How depth affects over and underfitting

Let us grow the tree deeper and deeper and watch the **training error** and the
**test error** (here the error is `1 - accuracy`). A very shallow tree is too simple and
does poorly on both (underfitting). As the tree gets deeper it fits the training data
better and better, but at some point it starts memorising noise: the training error keeps
falling toward zero while the test error stops improving and rises again (overfitting).

In [ ]:
depths = range(1, 21)
train_error, test_error = [], []

# 🎯 YOUR TURN — write the loop that fits one tree per depth and records its errors.
# 💭 Hints:
#    • You need a for-loop that runs once for each value in `depths`; give it a loop variable
#      (call it d) that holds the current depth on each pass.
#    • Inside, build a DecisionTreeClassifier whose max_depth is the current depth
#      (random_state=RANDOM_STATE) and .fit(...) it on the TRAIN set. Store it in `tree`.
#    • The two append lines are done for you: they turn accuracy into error (error = 1 - accuracy).
for ___ in depths:              # 🎯 replace ___ with your loop variable d
    tree = ...                  # 🎯 a DecisionTreeClassifier of depth d, fitted on X_train, y_train
    train_error.append(1 - tree.score(X_train, y_train))   # 1 - accuracy on train
    test_error.append(1 - tree.score(X_test, y_test))      # 1 - accuracy on test

In [ ]:
#@title Plot training vs test error { display-mode: "form" }
plt.figure(figsize=(7,4))
plt.plot(list(depths), train_error, marker='o', label='training error')
plt.plot(list(depths), test_error,  marker='s', label='test error')
plt.xlabel('tree depth (max_depth)'); plt.ylabel('classification error (1 - accuracy)')
plt.title('Decision tree: training vs test error as depth grows')
plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

The gap between the two curves is the price of overfitting: a single deep tree fits the
training data almost perfectly yet generalises poorly. A Random Forest tackles exactly
this problem by averaging many trees, which we look at next.

## Exercise 2. Random Forest

A Random Forest trains many trees on random subsets of the data and features, then
averages their votes. This bagging step cancels much of the variance of a single tree and
generalises far better.

**Task.**

1. Train a `RandomForestClassifier` with 300 trees and evaluate it.
2. Plot the feature importances and see which sensors matter.
3. Keep only the informative columns, retrain, and check how much performance changes.

scikit-learn references:
[RandomForestClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html),
[forests guide](https://scikit-learn.org/stable/modules/ensemble.html#forest),
[feature selection](https://scikit-learn.org/stable/modules/feature_selection.html).

In [ ]:
#@title 🌲 Visualization: how a Random Forest votes (double-click to view the code) { display-mode: "form" }
from IPython.display import HTML, display
import uuid as _uuid

_uid = "rf" + _uuid.uuid4().hex[:8]

_HTML = r"""
<style>
#__UID__{font-family:system-ui,Segoe UI,Roboto,sans-serif;background:linear-gradient(135deg,#f6f8ff,#fbf5ff);
  border:1px solid #ecebff;border-radius:18px;padding:18px 18px 20px;max-width:900px;color:#2c2350}
#__UID__ .rf-h{font-size:20px;font-weight:800;color:#3b2d6b;margin:0 0 4px}
#__UID__ .rf-sub{font-size:12.5px;color:#6b6685;line-height:1.5;margin:0 0 14px}
#__UID__ .rf-sub b{color:#3b2d6b}
#__UID__ .rf-top{display:flex;gap:14px;flex-wrap:wrap;align-items:stretch;margin-bottom:14px}
#__UID__ .rf-machine{flex:1 1 250px;background:#fff;border:1px solid #e6e8ee;border-radius:14px;padding:12px 14px}
#__UID__ .rf-mh{font-size:13px;font-weight:800;color:#2c2350;margin-bottom:8px;display:flex;align-items:center;gap:6px}
#__UID__ .rf-sensor{display:flex;align-items:center;gap:8px;margin:6px 0}
#__UID__ .rf-sname{font-size:11px;color:#6b6685;width:104px;flex:0 0 104px}
#__UID__ .rf-track{flex:1;height:9px;background:#eef0f5;border-radius:6px;overflow:hidden}
#__UID__ .rf-fill{height:100%;border-radius:6px;transition:width .5s ease}
#__UID__ .rf-controls{flex:1 1 200px;display:flex;flex-direction:column;justify-content:center;gap:10px;
  background:#fff;border:1px solid #e6e8ee;border-radius:14px;padding:14px}
#__UID__ .rf-btn{cursor:pointer;border:none;border-radius:10px;padding:11px 14px;font-size:13px;font-weight:700;
  color:#fff;background:linear-gradient(135deg,#667eea,#764ba2);box-shadow:0 6px 14px rgba(102,126,234,.32)}
#__UID__ .rf-btn.alt{background:linear-gradient(135deg,#8b86a6,#6b6685);box-shadow:none}
#__UID__ .rf-btn:active{transform:translateY(1px)}
#__UID__ .rf-grid{display:grid;grid-template-columns:repeat(3,1fr);gap:10px;margin-bottom:14px}
#__UID__ .rf-tree{background:#fff;border:1px solid #e6e8ee;border-radius:12px;padding:9px 10px;text-align:center;
  position:relative;opacity:.55;transition:opacity .25s,border-color .25s,box-shadow .25s}
#__UID__ .rf-tree.live{opacity:1;box-shadow:0 5px 13px rgba(102,126,234,.18)}
#__UID__ .rf-tree.fail.live{border-color:#e0796d}
#__UID__ .rf-tree.ok.live{border-color:#39b36a}
#__UID__ .rf-ticon{font-size:22px;line-height:1}
#__UID__ .rf-tname{font-size:10px;color:#6b6685;font-weight:700;margin-top:2px}
#__UID__ .rf-tfeat{font-size:9px;color:#a7a3bd;margin-top:3px;min-height:12px}
#__UID__ .rf-vote{margin-top:6px;font-size:11px;font-weight:800;border-radius:7px;padding:3px 0;
  background:#eef0f5;color:#c2c6d2;transition:all .25s}
#__UID__ .rf-tree.fail.live .rf-vote{background:#fdecea;color:#b5392a}
#__UID__ .rf-tree.ok.live .rf-vote{background:#eafaf0;color:#1f7a45}
#__UID__ .rf-agg{background:#fff;border:1px solid #e6e8ee;border-radius:14px;padding:13px 15px}
#__UID__ .rf-ah{font-size:12px;font-weight:800;color:#2c2350;margin-bottom:9px;display:flex;justify-content:space-between}
#__UID__ .rf-ah .rf-cnt{color:#6b6685;font-weight:600}
#__UID__ .rf-bar{height:26px;border-radius:8px;overflow:hidden;display:flex;background:#eef0f5}
#__UID__ .rf-bfail{height:100%;background:linear-gradient(90deg,#e0796d,#d4604f);width:0;transition:width .6s ease;
  display:flex;align-items:center;justify-content:center;color:#fff;font-size:11px;font-weight:800}
#__UID__ .rf-bok{height:100%;background:linear-gradient(90deg,#39b36a,#2f9e5d);width:0;transition:width .6s ease;
  display:flex;align-items:center;justify-content:center;color:#fff;font-size:11px;font-weight:800}
#__UID__ .rf-verdict{margin-top:11px;text-align:center;font-size:15px;font-weight:800;min-height:22px;color:#3b2d6b}
#__UID__ .rf-why{display:flex;gap:10px;flex-wrap:wrap;margin-top:14px}
#__UID__ .rf-chip{flex:1 1 220px;background:#fff;border:1px solid #ecebff;border-radius:11px;padding:9px 11px;
  font-size:11px;color:#6b6685;line-height:1.45}
#__UID__ .rf-chip b{color:#3b2d6b}
</style>
<div id="__UID__">
  <div class="rf-h">🌲 How a Random Forest reaches a verdict</div>
  <div class="rf-sub">A single tree is a nervous expert: it overreacts to its own training data (<b>high variance</b>).
  A forest asks <b>many trees</b>, each grown on a different <b>random sample of machines</b> (bootstrap) and allowed to
  look at only a <b>random subset of sensors</b> at each split. The individual trees disagree &mdash; but those errors
  are independent, so a <b>majority vote</b> cancels them out and the forest is far steadier.</div>

  <div class="rf-top">
    <div class="rf-machine">
      <div class="rf-mh">🛠️ Incoming machine</div>
      <div class="rf-sensor"><div class="rf-sname">🌡️ Temperature</div><div class="rf-track"><div class="rf-fill" data-s="temp"></div></div></div>
      <div class="rf-sensor"><div class="rf-sname">📳 Vibration</div><div class="rf-track"><div class="rf-fill" data-s="vib"></div></div></div>
      <div class="rf-sensor"><div class="rf-sname">⚡ Motor current</div><div class="rf-track"><div class="rf-fill" data-s="cur"></div></div></div>
      <div class="rf-sensor"><div class="rf-sname">🛢️ Oil level</div><div class="rf-track"><div class="rf-fill" data-s="oil"></div></div></div>
    </div>
    <div class="rf-controls">
      <button class="rf-btn" data-act="ask">🌲 Ask the forest</button>
      <button class="rf-btn alt" data-act="new">🎲 New machine</button>
    </div>
  </div>

  <div class="rf-grid"></div>

  <div class="rf-agg">
    <div class="rf-ah"><span>🗳️ The vote</span><span class="rf-cnt">press &ldquo;Ask the forest&rdquo;</span></div>
    <div class="rf-bar"><div class="rf-bfail"></div><div class="rf-bok"></div></div>
    <div class="rf-verdict">&nbsp;</div>
  </div>

  <div class="rf-why">
    <div class="rf-chip">🎯 <b>Randomness #1 &mdash; bootstrap rows.</b> Each tree trains on a random sample of machines (with
    replacement), so no two trees see the same data. Their mistakes point in different directions.</div>
    <div class="rf-chip">🎲 <b>Randomness #2 &mdash; random features.</b> At every split a tree may only choose from a random
    handful of sensors. This stops one strong sensor from dominating every tree and keeps the trees diverse.</div>
  </div>
</div>
<script>
(function(){
  var root = document.getElementById("__UID__");
  var FEATS = [
    {k:"temp", lab:"🌡️"},
    {k:"vib",  lab:"📳"},
    {k:"cur",  lab:"⚡"},
    {k:"oil",  lab:"🛢️"}
  ];
  var N = 9;            // trees in the forest
  var machine = {};
  var trees = [];
  var grid = root.querySelector(".rf-grid");
  var verdict = root.querySelector(".rf-verdict");
  var cnt = root.querySelector(".rf-cnt");
  var bfail = root.querySelector(".rf-bfail");
  var bok = root.querySelector(".rf-bok");

  function col(v){ // green -> amber -> red as risk grows
    if(v<0.5) return "#39b36a";
    if(v<0.7) return "#e0a23c";
    return "#e0796d";
  }
  function pick(arr,n){ // random subset of size n
    var c=arr.slice(), out=[];
    for(var i=0;i<n && c.length;i++){ out.push(c.splice(Math.floor(Math.random()*c.length),1)[0]); }
    return out;
  }
  function buildTrees(){
    trees=[]; grid.innerHTML="";
    for(var i=0;i<N;i++){
      var subset = pick(FEATS, 2 + Math.floor(Math.random()*2)); // 2-3 sensors
      var bias = (Math.random()-0.5)*0.42;                        // per-tree quirk = variance
      var el = document.createElement("div");
      el.className="rf-tree";
      el.innerHTML = '<div class="rf-ticon">🌳</div><div class="rf-tname">Tree '+(i+1)+'</div>'+
                     '<div class="rf-tfeat">sees '+subset.map(function(f){return f.lab;}).join(" ")+'</div>'+
                     '<div class="rf-vote">&middot;&middot;&middot;</div>';
      grid.appendChild(el);
      trees.push({el:el, subset:subset, bias:bias});
    }
  }
  function newMachine(){
    machine={temp:Math.random(), vib:Math.random(), cur:Math.random(), oil:Math.random()};
    root.querySelectorAll(".rf-fill").forEach(function(f){
      var v=machine[f.getAttribute("data-s")];
      var shown = f.getAttribute("data-s")==="oil" ? 1-v : v; // low oil = risky, so invert the bar
      f.style.width=(shown*100)+"%"; f.style.background=col(shown);
    });
    buildTrees(); reset();
  }
  function reset(){
    trees.forEach(function(t){
      t.el.classList.remove("live","fail","ok");
      t.el.querySelector(".rf-vote").innerHTML="&middot;&middot;&middot;";
    });
    bfail.style.width="0"; bok.style.width="0"; bfail.textContent=""; bok.textContent="";
    verdict.innerHTML="&nbsp;"; cnt.textContent="press “Ask the forest”";
  }
  function treeRisk(t){
    var s=0,w=0;
    t.subset.forEach(function(f){
      var v = f.k==="oil" ? 1-machine[f.k] : machine[f.k]; // low oil is risky
      s+=v; w++;
    });
    return (w? s/w : 0.5) + t.bias; // average of seen sensors + the tree's quirk
  }
  function ask(){
    reset();
    var fails=0, done=0;
    trees.forEach(function(t,i){
      setTimeout(function(){
        var fail = treeRisk(t) > 0.5;
        if(fail) fails++;
        t.el.classList.add("live", fail?"fail":"ok");
        t.el.querySelector(".rf-vote").textContent = fail ? "FAIL" : "OK";
        done++;
        if(done===N) tally(fails);
      }, 110*i + 120);
    });
  }
  function tally(fails){
    var oks=N-fails;
    bfail.style.width=(fails/N*100)+"%"; bok.style.width=(oks/N*100)+"%";
    bfail.textContent = fails? fails+" FAIL":""; bok.textContent = oks? oks+" OK":"";
    cnt.textContent = fails+" of "+N+" trees vote FAIL";
    setTimeout(function(){
      if(fails>oks) verdict.innerHTML='<span style="color:#b5392a">⚠️ Forest predicts: FAILURE</span> &nbsp;<span style="font-weight:600;color:#6b6685;font-size:12px">(majority rules)</span>';
      else if(oks>fails) verdict.innerHTML='<span style="color:#1f7a45">✅ Forest predicts: HEALTHY</span> &nbsp;<span style="font-weight:600;color:#6b6685;font-size:12px">(majority rules)</span>';
      else verdict.innerHTML='<span style="color:#3b2d6b">🤝 Split vote &mdash; a coin-flip tie</span>';
    }, 650);
  }
  root.querySelectorAll(".rf-btn").forEach(function(b){
    b.addEventListener("click", function(){
      if(b.getAttribute("data-act")==="new") newMachine(); else ask();
    });
  });
  newMachine();
})();
</script>
""".replace("__UID__", _uid)

display(HTML(_HTML))


### Your turn. Train

In [ ]:
# 🎯 YOUR TURN — Exercise 2: bagging, a forest of many trees.
#
# 💭 Think first: the single tree overfit once it grew deep. A Random Forest grows MANY deep trees,
#    each on a random subset of rows and features, then averages them. Notice the two deliberate
#    differences from Exercise 1:
#      • how many trees do we build?        -> set with n_estimators
#      • do we still cap the depth?         -> NO. We let the trees grow deep and rely on averaging
#                                              (bagging) to cancel the variance instead.
#
# 🎯 Implement: a RandomForestClassifier with 300 trees, n_jobs=-1 and random_state=RANDOM_STATE
#    (leave max_depth at its default, None). Fit it on X_train, y_train. Store it in `rf`.
rf = ...

# already written for you: the scoring
proba = rf.predict_proba(X_test)[:, 1]
auc = roc_auc_score(y_test, proba)
acc = accuracy_score(y_test, (proba >= 0.5).astype(int))
results['RandomForest'] = auc
print(f'Random Forest   ROC-AUC = {auc:.3f}   Accuracy = {acc:.3f}')

First, repeat the depth experiment with a forest, and compare it to the single tree from
Exercise 1 (we reuse the same `depths` and the tree `test_error`).

In [ ]:
rf_train_error, rf_test_error = [], []
for d in depths:                       # same depth range as the single tree above
    # 🎯 YOUR TURN — build and fit a forest of 100 trees, capped at the current depth d.
    # 💭 Hint: RandomForestClassifier(...) with n_estimators, max_depth, n_jobs=-1,
    #          random_state=RANDOM_STATE, then .fit(X_train, y_train). Store it in `forest`.
    forest = ...
    rf_train_error.append(1 - forest.score(X_train, y_train))
    rf_test_error.append(1 - forest.score(X_test, y_test))

In [ ]:
#@title Plot: forest vs single tree as depth grows { display-mode: "form" }
plt.figure(figsize=(7,4))
plt.plot(list(depths), rf_train_error, marker='o', label='forest training error')
plt.plot(list(depths), rf_test_error,  marker='s', label='forest test error')
plt.plot(list(depths), test_error, marker='^', linestyle='--', label='single tree test error')
plt.xlabel('tree depth (max_depth)'); plt.ylabel('classification error (1 - accuracy)')
plt.title('Random Forest vs a single tree'); plt.legend()
plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

Now the effect of the number of trees, with the depth left unlimited.

In [ ]:
n_trees = [1, 5, 10, 25, 50, 100, 200, 300]
ntrees_test_error = []
for n in n_trees:
    # 🎯 YOUR TURN — build and fit a forest with n trees (leave the depth at its default).
    # 💭 Hint: RandomForestClassifier(...) with n_estimators=n, n_jobs=-1, random_state=RANDOM_STATE,
    #          then .fit(X_train, y_train). Store it in `forest`.
    forest = ...
    ntrees_test_error.append(1 - forest.score(X_test, y_test))

In [ ]:
#@title Plot: effect of the number of trees { display-mode: "form" }
plt.figure(figsize=(7,4))
plt.plot(n_trees, ntrees_test_error, marker='o')
plt.xlabel('number of trees (n_estimators)'); plt.ylabel('test error (1 - accuracy)')
plt.title('More trees: error drops, then plateaus')
plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

### What the two experiments actually tell us

**Depth** has a sweet spot for a *single* tree (it overfits past depth&nbsp;6, error 0.24&nbsp;&rarr;&nbsp;0.30) but
not for the forest &mdash; its error stays flat near 0.21 even when every tree memorizes the training set perfectly.
Averaging cancels the overfitting that lives *inside* the trees.

**Number of trees** only helps, then plateaus: the first ~50 trees buy almost all the gain, and adding more never
hurts &mdash; a forest can't overfit by growing the ensemble.

Net: the forest (≈ 0.207) beats the best single tree (≈ 0.237) by ~3 points of accuracy, purely through
**variance reduction**, with no fragile depth to tune. Next we look at *which sensors* it leans on.

In [ ]:
#@title 📊 Visualization: the two experiments, side by side (double-click to view the code) { display-mode: "form" }
from IPython.display import HTML, display
import uuid as _uuid

_uid = "rfx" + _uuid.uuid4().hex[:8]

_HTML = r"""
<style>
#__UID__{font-family:system-ui,Segoe UI,Roboto,sans-serif;background:linear-gradient(135deg,#f6f8ff,#fbf5ff);
  border:1px solid #ecebff;border-radius:18px;padding:18px;max-width:900px;color:#2c2350;position:relative}
#__UID__ .rx-h{font-size:20px;font-weight:800;color:#3b2d6b;margin:0 0 12px}
#__UID__ .rx-row{display:flex;gap:14px;flex-wrap:wrap}
#__UID__ .rx-card{flex:1 1 380px;background:#fff;border:1px solid #e6e8ee;border-radius:14px;padding:12px 12px 8px}
#__UID__ .rx-ct{font-size:13px;font-weight:800;color:#2c2350;margin:0 0 2px}
#__UID__ .rx-cs{font-size:11px;color:#6b6685;margin:0 0 6px;line-height:1.4;min-height:30px}
#__UID__ .rx-leg{display:flex;gap:14px;flex-wrap:wrap;font-size:11px;color:#3b2d6b;margin:4px 2px 0;font-weight:600}
#__UID__ .rx-sw{display:inline-block;width:13px;height:3px;border-radius:2px;vertical-align:middle;margin-right:5px}
#__UID__ .rx-dot{cursor:pointer;transition:r .1s}
#__UID__ .rx-tip{position:absolute;pointer-events:none;background:#2c2350;color:#fff;font-size:11px;
  padding:4px 8px;border-radius:7px;opacity:0;transform:translate(-50%,-130%);white-space:nowrap;transition:opacity .1s;font-weight:600}
#__UID__ .rx-why{display:flex;gap:10px;flex-wrap:wrap;margin-top:14px}
#__UID__ .rx-chip{flex:1 1 250px;background:#fff;border:1px solid #ecebff;border-radius:11px;padding:9px 11px;
  font-size:11px;color:#6b6685;line-height:1.45}
#__UID__ .rx-chip b{color:#3b2d6b}
</style>
<div id="__UID__">
  <div class="rx-h">📊 Two knobs, two stories</div>
  <div class="rx-tip"></div>
  <div class="rx-row">
    <div class="rx-card">
      <div class="rx-ct">🌡️ Going deeper (100 trees, depth 1&rarr;20)</div>
      <div class="rx-cs">The single tree overfits past its sweet spot; the forest of the same deep trees stays flat.</div>
      <div class="rx-svg" data-chart="depth"></div>
      <div class="rx-leg">
        <span><span class="rx-sw" style="background:#e0796d"></span>single tree</span>
        <span><span class="rx-sw" style="background:#764ba2"></span>random forest</span>
      </div>
    </div>
    <div class="rx-card">
      <div class="rx-ct">🌲 Adding trees (depth unlimited, 1&rarr;300)</div>
      <div class="rx-cs">Error drops fast, then plateaus &mdash; and never climbs back. More trees can't overfit.</div>
      <div class="rx-svg" data-chart="trees"></div>
      <div class="rx-leg"><span><span class="rx-sw" style="background:#667eea"></span>forest test error</span></div>
    </div>
  </div>
  <div class="rx-why">
    <div class="rx-chip">🎯 <b>Depth has a sweet spot &mdash; for one tree only.</b> The lone tree's error rises after
    depth&nbsp;6 (0.24&rarr;0.30). The forest's barely moves, even though each tree hits <b>0.000 training error</b>.
    Averaging cancels the overfitting that lives inside the trees.</div>
    <div class="rx-chip">🌲 <b>More trees is "safer, just slower".</b> The first ~50 trees buy almost all the gain
    (0.35&rarr;0.21); after that it only steadies. You can't overfit by adding trees &mdash; so when in doubt, add more.</div>
    <div class="rx-chip">🏆 <b>Net result.</b> Best forest ≈ <b>0.207</b> vs best single tree ≈ 0.237 &mdash; ~3 pts of
    accuracy, won by <b>variance reduction</b>, with no fragile depth to tune.</div>
  </div>
</div>
<script>
(function(){
  var root = document.getElementById("__UID__");
  var tip  = root.querySelector(".rx-tip");

  // ---- real data measured from the experiments above ----
  var DEPTHS=[1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20];
  var TREE=[0.281,0.281,0.24,0.241,0.247,0.237,0.249,0.261,0.268,0.284,0.272,0.293,0.282,0.288,0.292,0.281,0.292,0.287,0.291,0.295];
  var RF=[0.252,0.232,0.23,0.219,0.223,0.227,0.218,0.21,0.215,0.221,0.214,0.22,0.222,0.211,0.207,0.217,0.212,0.214,0.213,0.213];
  var NT=[1,5,10,25,50,100,200,300];
  var NE=[0.351,0.276,0.271,0.235,0.214,0.211,0.215,0.212];

  var W=380,H=210,PADL=40,PADR=14,PADT=12,PADB=32,YMIN=0.18,YMAX=0.36;
  var iw=W-PADL-PADR, ih=H-PADT-PADB;
  function X(i,n){ return PADL + (n<2?0:i/(n-1)*iw); }
  function Y(v){ return PADT + (1-(v-YMIN)/(YMAX-YMIN))*ih; }

  function poly(data,color,dash){
    var pts=data.map(function(v,i){return X(i,data.length).toFixed(1)+","+Y(v).toFixed(1);}).join(" ");
    return '<polyline points="'+pts+'" fill="none" stroke="'+color+'" stroke-width="2.4" '+
           (dash?'stroke-dasharray="5 4" ':'')+'stroke-linejoin="round" stroke-linecap="round"/>';
  }
  function dots(data,color,xlabels,xname,sname){
    return data.map(function(v,i){
      return '<circle class="rx-dot" cx="'+X(i,data.length).toFixed(1)+'" cy="'+Y(v).toFixed(1)+'" r="3.2" '+
        'fill="#fff" stroke="'+color+'" stroke-width="2" '+
        'data-tip="'+xname+' '+xlabels[i]+' &bull; '+sname+': '+v.toFixed(3)+'"/>';
    }).join("");
  }
  function gridY(){
    var g="",vals=[0.20,0.25,0.30,0.35];
    vals.forEach(function(v){
      var y=Y(v).toFixed(1);
      g+='<line x1="'+PADL+'" y1="'+y+'" x2="'+(W-PADR)+'" y2="'+y+'" stroke="#eef0f5" stroke-width="1"/>'+
         '<text x="'+(PADL-6)+'" y="'+(Y(v)+3).toFixed(1)+'" text-anchor="end" font-size="9" fill="#a7a3bd">'+v.toFixed(2)+'</text>';
    });
    return g;
  }
  function xticks(labels,xname){
    var n=labels.length,g="",step=Math.ceil(n/8);
    for(var i=0;i<n;i++){ if(i%step!==0 && i!==n-1) continue;
      g+='<text x="'+X(i,n).toFixed(1)+'" y="'+(H-PADB+15)+'" text-anchor="middle" font-size="9" fill="#a7a3bd">'+labels[i]+'</text>';
    }
    g+='<text x="'+(PADL+iw/2)+'" y="'+(H-3)+'" text-anchor="middle" font-size="9.5" fill="#6b6685" font-weight="600">'+xname+'</text>';
    return g;
  }
  function vmark(i,n,label,color){
    var x=X(i,n).toFixed(1);
    return '<line x1="'+x+'" y1="'+PADT+'" x2="'+x+'" y2="'+(H-PADB)+'" stroke="'+color+'" stroke-width="1.2" stroke-dasharray="3 3" opacity="0.7"/>'+
           '<text x="'+x+'" y="'+(PADT+10)+'" text-anchor="middle" font-size="9" fill="'+color+'" font-weight="700">'+label+'</text>';
  }

  function depthSVG(){
    return '<svg viewBox="0 0 '+W+' '+H+'" width="100%" style="overflow:visible">'+
      gridY()+
      vmark(5,DEPTHS.length,'sweet spot','#c2876a')+   // depth 6 = index 5
      poly(TREE,'#e0796d',true)+poly(RF,'#764ba2',false)+
      dots(TREE,'#e0796d',DEPTHS,'depth','single tree')+
      dots(RF,'#764ba2',DEPTHS,'depth','forest')+
      xticks(DEPTHS,'tree depth (max_depth)')+
      '</svg>';
  }
  function treesSVG(){
    return '<svg viewBox="0 0 '+W+' '+H+'" width="100%" style="overflow:visible">'+
      gridY()+
      vmark(4,NT.length,'plateau','#5b6fb0')+          // 50 trees = index 4
      poly(NE,'#667eea',false)+
      dots(NE,'#667eea',NT,'trees','forest')+
      xticks(NT,'number of trees (n_estimators)')+
      '</svg>';
  }
  root.querySelector('[data-chart="depth"]').innerHTML=depthSVG();
  root.querySelector('[data-chart="trees"]').innerHTML=treesSVG();

  // hover tooltips
  root.querySelectorAll(".rx-dot").forEach(function(d){
    d.addEventListener("mouseenter",function(e){
      var r=root.getBoundingClientRect(), b=d.getBoundingClientRect();
      tip.innerHTML=d.getAttribute("data-tip");
      tip.style.left=(b.left-r.left+b.width/2)+"px";
      tip.style.top=(b.top-r.top)+"px";
      tip.style.opacity="1"; d.setAttribute("r","5");
    });
    d.addEventListener("mouseleave",function(){ tip.style.opacity="0"; d.setAttribute("r","3.2"); });
  });
})();
</script>
""".replace("__UID__", _uid)

display(HTML(_HTML))


### Your turn. Feature importance

**Tools for this one** — it fits on a single line:

- `rf.feature_importances_` — a NumPy array with one importance per column.
- `X_train.columns` — the matching column names.
- `pd.Series(values, index=names)` — pairs each value with its name into a labelled Series.
- `.sort_values(ascending=False)` — orders that Series from most to least important.

Put these together to build `imp` (no loop needed).

In [ ]:
# 🎯 YOUR TURN — which sensors did the forest actually rely on?
#
# A fitted forest exposes `rf.feature_importances_`: one score per column, all summing to 1.
# 💭 Think first: the data hides 35 pure-noise columns (named sensor_noise_*). Before you run it —
#    where do you expect them to land in the ranking, near the top or near the bottom?
#
# 🎯 Implement: build a pandas Series from rf.feature_importances_, indexed by X_train.columns and
#    sorted in descending order. Store it in `imp`.
imp = ...
imp.head(10)

In [ ]:
#@title Plot feature importance { display-mode: "form" }
imp.head(12).plot.barh(figsize=(7,4)).invert_yaxis()
plt.title('Random Forest feature importance (top 12)'); plt.tight_layout(); plt.show()

A handful of sensors dominate, while the many `sensor_noise_*` columns get almost zero
importance. That is exactly the signal we use for feature selection.

### ⭐ Bonus. Grid search over `max_depth` &times; `n_estimators`

So far we changed one knob at a time. In practice you tune them **together** and let
cross-validation pick the winner &mdash; never the test set. A `GridSearchCV` trains a forest for
every `(max_depth, n_estimators)` combination, scores each with k-fold CV on the *training*
data, and reports the best. Below we sweep a small grid and draw the resulting CV-AUC surface
as a heatmap.

In [ ]:
from sklearn.model_selection import GridSearchCV

# Two hyperparameters, searched jointly with cross-validation (the test set is never touched).
depth_grid = [3, 5, 8, 12, None]      # None = grow each tree until its leaves are pure
ntree_grid = [50, 100, 200, 300]

# 🎯 YOUR TURN — set up and run the grid search. Store the fitted search in `grid`.
# 💭 Hints — GridSearchCV(estimator, param_grid, scoring=..., cv=..., n_jobs=-1):
#    • estimator   = a RandomForestClassifier(n_jobs=-1, random_state=RANDOM_STATE)
#    • param_grid  = a dict mapping each name to its list: {'max_depth': depth_grid, 'n_estimators': ntree_grid}
#    • scoring='roc_auc',  cv=3   (3-fold cross-validation, on the TRAIN set only)
#    • finish the call with .fit(X_train, y_train)
grid = ...

# already written for you: reshape the mean CV score into a (depth x n_trees) matrix for the heatmap
auc_grid = np.full((len(depth_grid), len(ntree_grid)), np.nan)
for p, s in zip(grid.cv_results_['params'], grid.cv_results_['mean_test_score']):
    auc_grid[depth_grid.index(p['max_depth']), ntree_grid.index(p['n_estimators'])] = s

print('Best params :', grid.best_params_)
print('Best CV AUC : {:.3f}'.format(grid.best_score_))
print('Grid spread : {:.3f}  (max - min CV AUC across the whole grid)'.format(
    np.nanmax(auc_grid) - np.nanmin(auc_grid)))

In [ ]:
#@title 📊 Visualization: the grid-search heatmap (double-click to view the code) { display-mode: "form" }
from IPython.display import HTML, display

def _rf_grid_heatmap(auc_grid, depth_grid, ntree_grid):
    vmin, vmax = float(np.nanmin(auc_grid)), float(np.nanmax(auc_grid))
    span = (vmax - vmin) or 1.0
    LO, HI = (238, 243, 255), (118, 75, 162)            # #eef3ff -> #764ba2
    bi = np.unravel_index(np.nanargmax(auc_grid), auc_grid.shape)
    dlab = lambda d: '∞' if d is None else str(d)   # None -> infinity sign

    head = ''.join(f'<th>{n}</th>' for n in ntree_grid)
    rows = ''
    for r, d in enumerate(depth_grid):
        cells = ''
        for c, n in enumerate(ntree_grid):
            v = float(auc_grid[r, c]); t = (v - vmin) / span
            cr, cg, cb = (round(LO[i] + (HI[i] - LO[i]) * t) for i in range(3))
            txt = '#fff' if t > 0.55 else '#3b2d6b'
            best = (r, c) == bi
            star = '⭐ ' if best else ''
            bd = 'box-shadow:0 0 0 3px #e6b400 inset;font-weight:800;' if best else ''
            cells += (f'<td style="background:rgb({cr},{cg},{cb});color:{txt};{bd}" '
                      f'title="depth={dlab(d)}, trees={n} → CV AUC {v:.3f}">{star}{v:.3f}</td>')
        rows += f'<tr><th class="rg-rh">{dlab(d)}</th>{cells}</tr>'

    bd_, bn_ = depth_grid[bi[0]], ntree_grid[bi[1]]
    cap = (f'\U0001f3c6 Best: <b>depth {dlab(bd_)}</b>, <b>{bn_} trees</b> &rarr; CV AUC <b>{vmax:.3f}</b>. '
           f'The whole grid spans only <b>{vmax - vmin:.3f}</b> AUC &mdash; a forest is forgiving: once the trees '
           f'are deep enough and you have &ge;100 of them, extra tuning barely moves the needle. The only clearly '
           f'weak corner is shallow trees (depth&nbsp;3).')

    style = '''
    <style>
    .rg{font-family:system-ui,Segoe UI,Roboto,sans-serif;background:linear-gradient(135deg,#f6f8ff,#fbf5ff);
        border:1px solid #ecebff;border-radius:18px;padding:18px;max-width:760px;color:#2c2350}
    .rg-h{font-size:20px;font-weight:800;color:#3b2d6b;margin:0 0 12px}
    .rg-wrap{display:flex;gap:12px;align-items:center;flex-wrap:wrap}
    .rg-yl{writing-mode:vertical-rl;transform:rotate(180deg);font-size:11px;font-weight:700;color:#6b6685}
    .rg-t{border-collapse:separate;border-spacing:4px}
    .rg-t th{font-size:11px;color:#6b6685;font-weight:700;padding:2px 6px}
    .rg-t td{width:70px;height:42px;text-align:center;border-radius:9px;font-size:13px;font-weight:700}
    .rg-t th.rg-rh{color:#3b2d6b;font-weight:800;text-align:right}
    .rg-xl{text-align:center;font-size:11px;font-weight:700;color:#6b6685;margin-top:4px}
    .rg-bar{height:10px;border-radius:6px;background:linear-gradient(90deg,#eef3ff,#764ba2);flex:0 0 120px}
    .rg-leg{display:flex;align-items:center;gap:8px;font-size:11px;color:#6b6685;margin-top:10px}
    .rg-cap{font-size:11.5px;color:#6b6685;line-height:1.5;margin-top:12px;background:#fff;border:1px solid #ecebff;
            border-radius:11px;padding:9px 11px}
    .rg-cap b{color:#3b2d6b}
    </style>'''

    html = style + f'''
    <div class="rg">
      <div class="rg-h">\U0001f5fa️ Cross-validated AUC for every (depth, trees) pair</div>
      <div class="rg-wrap">
        <div class="rg-yl">max_depth &darr;</div>
        <table class="rg-t">
          <tr><th></th><th colspan="{len(ntree_grid)}" style="color:#3b2d6b">n_estimators &rarr;</th></tr>
          <tr><th></th>{head}</tr>
          {rows}
        </table>
      </div>
      <div class="rg-leg">low&nbsp;AUC<span class="rg-bar"></span>high&nbsp;AUC
        &nbsp;&nbsp;<span style="color:#e6b400;font-weight:800">⭐</span> best on the grid</div>
      <div class="rg-cap">{cap}</div>
    </div>'''
    display(HTML(html))

_rf_grid_heatmap(auc_grid, depth_grid, ntree_grid)


### Your turn. Feature selection

Keep only the columns whose importance is clearly above the noise level, then retrain.

In [ ]:
# already written for you: the noise level (the largest importance among the sensor_noise_* columns)
noise_floor = imp[[c for c in imp.index if c.startswith('sensor_noise_')]].max()

# 💭 Think first: a genuinely useful sensor should be MORE important than the best pure-noise column.
#    So we keep only the columns whose importance is above `noise_floor`, then retrain on those.
#    Predict the outcome: will dropping ~40 columns make the score drop a lot, a little, or not at all?
#
# Reminder — to keep the names of a Series whose value is above a threshold, filter the index:
#     s = pd.Series({'a': 0.5, 'b': 0.1, 'c': 0.3})
#     threshold = 0.2
#     kept = [name for name in s.index if s[name] > threshold]   # -> ['a', 'c']
#
# 🎯 Implement (1): keep = the columns of `imp` whose importance is strictly above noise_floor.
keep = ...
# 🎯 Implement (2): rf_sel = a RandomForestClassifier with the SAME settings as before (300 trees,
#     n_jobs=-1, random_state=RANDOM_STATE), fit on the reduced data X_train[keep], y_train.
rf_sel = ...

# already written for you: compare the two models
print(f'Keeping {len(keep)} of {X.shape[1]} columns')
auc_all = roc_auc_score(y_test, rf.predict_proba(X_test)[:,1])
auc_sel = roc_auc_score(y_test, rf_sel.predict_proba(X_test[keep])[:,1])
print(f'all features : ROC-AUC={auc_all:.4f}')
print(f'selected     : ROC-AUC={auc_sel:.4f}  ({len(keep)} columns)')

We dropped most columns and kept the same performance. A simpler model is cheaper to run
and easier to explain, which matters a lot for a risk model in production.

In [ ]:
#@title 📊 Visualization: all features vs selected features (double-click to view the code) { display-mode: "form" }
from IPython.display import HTML, display

def _rf_feature_compare(auc_all, auc_sel, n_all, n_sel):
    def w(v, lo=0.5, hi=1.0):                       # AUC -> bar width %, honest [0.5,1.0] scale
        return max(0.0, min(100.0, (v - lo) / (hi - lo) * 100))
    dcols = n_all - n_sel
    pct = dcols / n_all * 100 if n_all else 0
    dauc = auc_sel - auc_all
    win = dauc >= -0.002                            # selected is as good or better
    sign = '+' if dauc >= 0 else '−'
    sel_accent = '#39b36a' if win else '#e0796d'

    def panel(title, n, auc, accent, tag):
        return (f'<div class="fc-card" style="border-color:{accent}55">'
                f'<div class="fc-tag" style="background:{accent}1a;color:{accent}">{tag}</div>'
                f'<div class="fc-t">{title}</div>'
                f'<div class="fc-n" style="color:{accent}">{n}</div><div class="fc-nl">columns used</div>'
                f'<div class="fc-track"><div class="fc-fill" style="width:{w(auc):.1f}%;background:{accent}"></div></div>'
                f'<div class="fc-auc">ROC-AUC <b>{auc:.4f}</b></div></div>')

    style = '''
    <style>
    .fc{font-family:system-ui,Segoe UI,Roboto,sans-serif;background:linear-gradient(135deg,#f6f8ff,#fbf5ff);
        border:1px solid #ecebff;border-radius:18px;padding:18px;max-width:760px;color:#2c2350}
    .fc-h{font-size:20px;font-weight:800;color:#3b2d6b;margin:0 0 12px}
    .fc-row{display:flex;align-items:center;gap:12px;flex-wrap:wrap}
    .fc-card{flex:1 1 210px;background:#fff;border:1px solid #e6e8ee;border-radius:14px;padding:13px 15px;position:relative}
    .fc-tag{position:absolute;top:11px;right:11px;font-size:10px;font-weight:800;border-radius:7px;padding:2px 7px}
    .fc-t{font-size:13px;font-weight:800;color:#2c2350;margin-bottom:6px}
    .fc-n{font-size:34px;font-weight:800;line-height:1}
    .fc-nl{font-size:11px;color:#8b86a6;margin-bottom:10px}
    .fc-track{height:10px;background:#eef0f5;border-radius:6px;overflow:hidden}
    .fc-fill{height:100%;border-radius:6px;transition:width .6s ease}
    .fc-auc{font-size:12px;color:#6b6685;margin-top:6px}.fc-auc b{color:#3b2d6b;font-size:13px}
    .fc-arrow{font-size:24px;color:#b9a9e6;flex:0 0 auto}
    .fc-cap{font-size:11.5px;color:#6b6685;line-height:1.5;margin-top:13px;background:#fff;border:1px solid #ecebff;
            border-radius:11px;padding:9px 11px}.fc-cap b{color:#3b2d6b}
    .fc-delta{display:inline-block;font-weight:800;border-radius:7px;padding:2px 8px;font-size:12px}
    </style>'''

    html = style + f'''
    <div class="fc">
      <div class="fc-h">✂️ Fewer features, same signal</div>
      <div class="fc-row">
        {panel("All features", n_all, auc_all, "#8b86a6", "baseline")}
        <div class="fc-arrow">&rarr;</div>
        {panel("Selected only", n_sel, auc_sel, sel_accent, "✔ kept" if win else "kept")}
      </div>
      <div class="fc-cap">
        Dropped <b>{dcols} of {n_all}</b> columns (<b>{pct:.0f}%</b>, almost all the
        <code>sensor_noise_*</code> junk) and the AUC
        <span class="fc-delta" style="background:{sel_accent}1a;color:{sel_accent}">{sign}{abs(dauc):.4f}</span>
        &mdash; {"it even ticked up" if dauc > 0 else "held steady"} ({auc_all:.4f} &rarr; {auc_sel:.4f}).
        A model on <b>{n_sel}</b> sensors is cheaper to run, needs fewer instruments to maintain, and is far easier
        to trust and explain &mdash; which matters a lot for a risk model in production.
      </div>
    </div>'''
    display(HTML(html))

_rf_feature_compare(auc_all, auc_sel, X.shape[1], len(keep))


## Exercise 3. Gradient Boosting with XGBoost and LightGBM

A forest trains all its trees **independently, in parallel**, then averages them. Boosting does
the opposite: it builds trees **one at a time, in a chain**, and each new tree's only job is to fix
what the chain *so far* still gets wrong. Each round:

1. Run the model we have and measure **how wrong it still is on every machine** &mdash; the *residual*:
   how far off the prediction is, and in which direction.
2. Train a small new tree to **predict that leftover error**.
3. Add it to the model, but only a fraction of it &mdash; the **learning rate** (say 0.05) takes a small,
   safe step instead of overcorrecting on one round.
4. Repeat for hundreds of rounds. Each tree is weak on its own (just a few splits), but the running
   sum becomes a sharp predictor &mdash; exactly the picture above, where many crude single cuts add up
   to one tight decision boundary.

**Why this beats a forest on tabular data.** A tree makes two kinds of error. **Variance** &mdash; it is
jumpy, so a small change in the data swings its predictions. **Bias** &mdash; it systematically misses
real structure. A forest averages many deep trees, which cancels the *variance* but cannot remove a
bias they all share. Boosting attacks the **bias** head-on: every new tree points straight at the
remaining error, so the model keeps closing systematic gaps a forest would just average over.

That same targeting is boosting's risk. Because each tree chases the leftover error, **adding too many
trees eventually fits the noise** &mdash; so boosting has to be reined in with the **learning rate**, the
**tree depth** (kept shallow), and the **number of trees**. (A forest, remember, *can't* overfit just by
adding trees.) **XGBoost** and **LightGBM** are two fast, heavily optimized versions of this idea with
built-in regularization to keep that overfitting in check &mdash; usually the top performers on
structured/tabular data.

**Task.** Train an XGBoost and a LightGBM classifier and compare them with the forest.

References:
[scikit-learn gradient boosting](https://scikit-learn.org/stable/modules/ensemble.html#gradient-boosting),
[XGBClassifier](https://xgboost.readthedocs.io/en/stable/python/python_api.html#xgboost.XGBClassifier),
[LGBMClassifier](https://lightgbm.readthedocs.io/en/latest/pythonapi/lightgbm.LGBMClassifier.html).

In [ ]:
#@title 🚀 Visualization: individual trees vs the combined boosted model (double-click to view the code) { display-mode: "form" }
from IPython.display import HTML, display
import uuid as _uuid

_uid = "bst" + _uuid.uuid4().hex[:8]

_HTML = r"""
<style>
#__UID__{font-family:system-ui,Segoe UI,Roboto,sans-serif;background:linear-gradient(135deg,#f6f8ff,#fbf5ff);
  border:1px solid #ecebff;border-radius:18px;padding:18px;max-width:900px;color:#2c2350}
#__UID__ .bx-h{font-size:20px;font-weight:800;color:#3b2d6b;margin:0 0 4px}
#__UID__ .bx-sub{font-size:12.5px;color:#6b6685;line-height:1.55;margin:0 0 14px}
#__UID__ .bx-sub b{color:#3b2d6b}
#__UID__ .bx-row{display:flex;gap:14px;flex-wrap:wrap;align-items:stretch}
#__UID__ .bx-main{flex:1 1 330px;background:#fff;border:1px solid #e6e8ee;border-radius:14px;padding:11px 13px}
#__UID__ .bx-side{flex:1 1 230px;display:flex;flex-direction:column;gap:10px}
#__UID__ .bx-card{background:#fff;border:1px solid #e6e8ee;border-radius:14px;padding:12px 14px}
#__UID__ .bx-ct{font-size:13px;font-weight:800;color:#2c2350;margin:0 0 1px}
#__UID__ .bx-cs{font-size:11px;color:#6b6685;margin:0 0 6px;line-height:1.4}
#__UID__ .bx-accwrap{display:flex;align-items:center;gap:9px}
#__UID__ .bx-track{flex:1;height:14px;background:#eef0f5;border-radius:8px;overflow:hidden}
#__UID__ .bx-fill{height:100%;width:0;border-radius:8px;background:linear-gradient(90deg,#39b36a,#2f9e5d);transition:width .5s ease}
#__UID__ .bx-accv{font-size:14px;font-weight:800;color:#1f7a45;width:46px;text-align:right}
#__UID__ .bx-btns{display:flex;gap:8px;margin-top:11px}
#__UID__ .bx-btn{cursor:pointer;border:none;border-radius:10px;padding:10px 12px;font-size:12.5px;font-weight:700;
  color:#fff;background:linear-gradient(135deg,#667eea,#764ba2);box-shadow:0 6px 14px rgba(102,126,234,.32);flex:1}
#__UID__ .bx-btn.alt{background:linear-gradient(135deg,#8b86a6,#6b6685);box-shadow:none;flex:0 0 auto}
#__UID__ .bx-btn:active{transform:translateY(1px)}
#__UID__ .bx-btn:disabled{opacity:.45;cursor:default;transform:none}
#__UID__ .bx-cap{font-size:11.5px;color:#6b6685;line-height:1.5}
#__UID__ .bx-cap b{color:#3b2d6b}
#__UID__ .bx-leg{font-size:10.5px;color:#8b86a6;margin-top:6px;display:flex;gap:13px;flex-wrap:wrap}
#__UID__ .bx-striph{font-size:13px;font-weight:800;color:#2c2350;margin:16px 0 3px}
#__UID__ .bx-strsub{font-size:11px;color:#6b6685;margin:0 0 8px;line-height:1.4}
#__UID__ .bx-strip{display:flex;gap:9px;flex-wrap:wrap}
#__UID__ .bx-tree{background:#fff;border:1px solid #e6e8ee;border-radius:11px;padding:6px;text-align:center;width:84px}
#__UID__ .bx-tree.new{border-color:#764ba2;box-shadow:0 4px 11px rgba(118,75,162,.22)}
#__UID__ .bx-tl{font-size:9.5px;color:#6b6685;font-weight:700;margin-top:3px}
#__UID__ .bx-tv{font-size:9px;color:#8b86a6}
#__UID__ .bx-empty{font-size:11px;color:#a7a3bd;padding:8px}
#__UID__ .bx-why{display:flex;gap:10px;flex-wrap:wrap;margin-top:14px}
#__UID__ .bx-w{flex:1 1 250px;background:#fff;border:1px solid #ecebff;border-radius:11px;padding:9px 11px;
  font-size:11px;color:#6b6685;line-height:1.45}
#__UID__ .bx-w b{color:#3b2d6b}
</style>
<div id="__UID__">
  <div class="bx-h">🚀 One weak tree at a time &mdash; combined into one strong model</div>
  <div class="bx-sub">Each machine is a dot: 🔴 will fail, 🟢 healthy. A single boosting tree is <b>weak</b> &mdash; it can
  only make <b>one straight cut</b> (see the strip below). But boosting adds them <b>in sequence</b>, each new tree
  aimed at the dots still <b>ringed as wrong</b>, and sums their weighted votes. The big panel is that
  <b>combined model</b>: watch its red/green regions wrap around the true shape as trees pile up.</div>

  <div class="bx-row">
    <div class="bx-main">
      <div class="bx-ct">🌍 The combined boosted model</div>
      <div class="bx-cs">Shaded = where the ensemble predicts fail (red) vs healthy (green). Dots = the real labels.</div>
      <svg class="bx-svgM" viewBox="0 0 300 300" width="100%" style="overflow:visible"></svg>
      <div class="bx-leg"><span>🔴 fail &nbsp; 🟢 healthy</span><span>◌ ring = ensemble still wrong</span></div>
    </div>
    <div class="bx-side">
      <div class="bx-card">
        <div class="bx-ct">🎯 Combined accuracy</div>
        <div class="bx-accwrap"><div class="bx-track"><div class="bx-fill"></div></div><div class="bx-accv">—</div></div>
        <div class="bx-btns">
          <button class="bx-btn" data-act="next">▶ Add a tree</button>
          <button class="bx-btn" data-act="auto">⏩ Auto</button>
          <button class="bx-btn alt" data-act="reset">↺</button>
        </div>
      </div>
      <div class="bx-card"><div class="bx-cap">Press <b>▶ Add a tree</b>. The first tree alone is barely better than guessing &mdash; the power comes from stacking them.</div></div>
    </div>
  </div>

  <div class="bx-striph">🌳 The individual trees (each makes just ONE cut)</div>
  <div class="bx-strsub">Every thumbnail is a whole tree on its own: one split, a red &ldquo;fail&rdquo; side and a green &ldquo;healthy&rdquo; side. Its <i>vote</i> is how much it counts in the sum.</div>
  <div class="bx-strip"></div>

  <div class="bx-why">
    <div class="bx-w">🌲 <b>Bagging (forest):</b> many deep, independent trees in parallel, then averaged &mdash; mainly cuts <b>variance</b>.</div>
    <div class="bx-w">🚀 <b>Boosting (XGBoost / LightGBM):</b> many shallow trees in sequence, each fixing the previous errors &mdash; cuts <b>bias too</b>, usually the best single model on tabular data.</div>
  </div>
</div>
<script>
(function(){
  var root=document.getElementById("__UID__");
  var sM=root.querySelector(".bx-svgM"), strip=root.querySelector(".bx-strip");
  var fill=root.querySelector(".bx-fill"), accv=root.querySelector(".bx-accv"), cap=root.querySelector(".bx-cap");
  var bN=root.querySelector('[data-act="next"]'), bA=root.querySelector('[data-act="auto"]'), bR=root.querySelector('[data-act="reset"]');
  var SVGNS="http://www.w3.org/2000/svg", M=300, P=10, pw=M-2*P, G=22, MAXR=12;
  var pts=[], w=[], learners=[], timer=null;
  var RED="#e0796d", GRN="#39b36a";

  function lcg(s){return function(){s=(s*1664525+1013904223)>>>0;return s/4294967296;};}
  function genPoints(){
    var rng=lcg(20260627), p=[];
    while(p.length<60){
      var x=rng(), y=rng(), d=Math.hypot(x-0.5,y-0.5);
      if(Math.abs(d-0.30)<0.04) continue;            // clear margin around the boundary
      p.push({x:x,y:y,label:(d<0.30)?1:-1});          // inside the disk = will fail
    }
    [5,17,29,41,53].forEach(function(i){p[i].label*=-1;}); // a little label noise
    return p;
  }
  function PX(x){return P+x*pw;}
  function PY(y){return P+(1-y)*pw;}
  function hAt(L,x,y){return L.pol*(((L.ax==="x"?x:y)>L.thr)?1:-1);}
  function hOf(L,p){return hAt(L,p.x,p.y);}
  function Fat(x,y){var F=0;for(var k=0;k<learners.length;k++)F+=learners[k].alpha*hAt(learners[k],x,y);return F;}
  function ensPred(p){return Fat(p.x,p.y)>0?1:-1;}
  function accuracy(){var c=0;for(var i=0;i<pts.length;i++)if(ensPred(pts[i])===pts[i].label)c++;return c/pts.length;}
  function alphaSum(){var s=0;for(var k=0;k<learners.length;k++)s+=Math.abs(learners[k].alpha);return s||1;}

  function bestStump(){
    var best=null;
    ["x","y"].forEach(function(ax){
      for(var t=0.06;t<=0.94;t+=0.04){
        [1,-1].forEach(function(pol){
          var e=0;
          for(var i=0;i<pts.length;i++){var h=pol*((pts[i][ax]>t)?1:-1);if(h!==pts[i].label)e+=w[i];}
          if(!best||e<best.e)best={ax:ax,thr:t,pol:pol,e:e};
        });
      }
    });
    return best;
  }
  function el(n,a){var e=document.createElementNS(SVGNS,n);for(var k in a)e.setAttribute(k,a[k]);return e;}

  function drawMain(){
    sM.innerHTML="";
    var Fn=alphaSum(), cw=pw/G;
    if(learners.length){
      for(var gx=0;gx<G;gx++)for(var gy=0;gy<G;gy++){
        var cx=(gx+0.5)/G, cy=(gy+0.5)/G, F=Fat(cx,cy), conf=Math.min(1,Math.abs(F)/Fn);
        var rc=el("rect",{x:(P+gx*cw).toFixed(1),y:(P+(G-1-gy)*cw).toFixed(1),width:cw+0.6,height:cw+0.6,
          fill:(F>0?RED:GRN)});
        rc.setAttribute("opacity",(0.07+0.34*conf).toFixed(3)); sM.appendChild(rc);
      }
    } else {
      sM.appendChild(el("rect",{x:P,y:P,width:pw,height:pw,fill:"#fbfcff",stroke:"#eef0f5",rx:8}));
    }
    sM.appendChild(el("rect",{x:P,y:P,width:pw,height:pw,fill:"none",stroke:"#e6e8ee",rx:8}));
    for(var i=0;i<pts.length;i++){
      var p=pts[i], wrong=learners.length && ensPred(p)!==p.label;
      if(wrong){var ring=el("circle",{cx:PX(p.x),cy:PY(p.y),r:7.5,fill:"none",stroke:"#2c2350","stroke-width":1.7});
        ring.setAttribute("opacity","0.6");sM.appendChild(ring);}
      sM.appendChild(el("circle",{cx:PX(p.x),cy:PY(p.y),r:4.5,fill:(p.label>0?RED:GRN),stroke:"#fff","stroke-width":1.4}));
    }
  }

  function thumb(L){
    var T=64, q=4, iw=T-2*q;
    function tx(x){return q+x*iw;} function ty(y){return q+(1-y)*iw;}
    var failRect, okRect, line;
    if(L.ax==="x"){var xt=tx(L.thr);
      var failLeft=(L.pol<0);
      failRect=failLeft?{x:q,y:q,w:xt-q,h:iw}:{x:xt,y:q,w:q+iw-xt,h:iw};
      okRect  =failLeft?{x:xt,y:q,w:q+iw-xt,h:iw}:{x:q,y:q,w:xt-q,h:iw};
      line='<line x1="'+xt.toFixed(1)+'" y1="'+q+'" x2="'+xt.toFixed(1)+'" y2="'+(q+iw)+'" stroke="#3b2d6b" stroke-width="1.4"/>';
    } else {var yt=ty(L.thr);
      var failTop=(L.pol>0);
      failRect=failTop?{x:q,y:q,w:iw,h:yt-q}:{x:q,y:yt,w:iw,h:q+iw-yt};
      okRect  =failTop?{x:q,y:yt,w:iw,h:q+iw-yt}:{x:q,y:q,w:iw,h:yt-q};
      line='<line x1="'+q+'" y1="'+yt.toFixed(1)+'" x2="'+(q+iw)+'" y2="'+yt.toFixed(1)+'" stroke="#3b2d6b" stroke-width="1.4"/>';
    }
    function rk(r,c){return '<rect x="'+r.x.toFixed(1)+'" y="'+r.y.toFixed(1)+'" width="'+Math.max(0,r.w).toFixed(1)+'" height="'+Math.max(0,r.h).toFixed(1)+'" fill="'+c+'" opacity="0.5"/>';}
    return '<svg viewBox="0 0 '+T+' '+T+'" width="100%">'+rk(failRect,RED)+rk(okRect,GRN)+line+
           '<rect x="'+q+'" y="'+q+'" width="'+iw+'" height="'+iw+'" fill="none" stroke="#e6e8ee"/></svg>';
  }
  function drawStrip(){
    if(!learners.length){strip.innerHTML='<div class="bx-empty">No trees yet &mdash; press &ldquo;Add a tree&rdquo;.</div>';return;}
    strip.innerHTML=learners.map(function(L,k){
      var op=L.ax+(L.pol>0?" > ":" ≤ ")+L.thr.toFixed(2);
      var cls="bx-tree"+(k===learners.length-1?" new":"");
      return '<div class="'+cls+'">'+thumb(L)+'<div class="bx-tl">#'+(k+1)+': '+op+'</div><div class="bx-tv">vote '+L.alpha.toFixed(2)+'</div></div>';
    }).join("");
  }
  function refresh(){
    var has=learners.length>0, acc=has?accuracy():0;
    fill.style.width=has?(acc*100)+"%":"0"; accv.textContent=has?Math.round(acc*100)+"%":"—";
    bN.disabled=learners.length>=MAXR;
    if(learners.length>=MAXR){bN.textContent="✓ "+MAXR+" trees";stopAuto();bA.disabled=true;}
  }
  function addTree(){
    if(learners.length>=MAXR) return;
    var s=bestStump(), e=Math.min(Math.max(s.e,1e-6),0.5-1e-6);
    s.alpha=0.5*Math.log((1-e)/e);
    var Z=0;
    for(var i=0;i<pts.length;i++){w[i]*=Math.exp(-s.alpha*pts[i].label*hOf(s,pts[i]));Z+=w[i];}
    for(var i=0;i<pts.length;i++)w[i]/=Z;
    learners.push(s);
    drawMain(); drawStrip(); refresh();
    cap.innerHTML='<b>'+learners.length+' tree'+(learners.length>1?'s':'')+' combined.</b> The newest one cut on <b>'+
      s.ax+(s.pol>0?" &gt; ":" &le; ")+s.thr.toFixed(2)+'</b>, focusing on the ringed mistakes. On its own each cut is crude, '+
      'but the weighted sum already gets <b>'+Math.round(accuracy()*100)+'%</b> right &mdash; and the region keeps tightening around the true shape.';
  }
  function stopAuto(){if(timer){clearInterval(timer);timer=null;bA.textContent="⏩ Auto";}}
  function toggleAuto(){if(timer){stopAuto();return;}bA.textContent="⏸";timer=setInterval(function(){if(learners.length>=MAXR){stopAuto();return;}addTree();},700);}
  function reset(){
    stopAuto(); bA.disabled=false;
    pts=genPoints(); w=pts.map(function(){return 1/pts.length;}); learners=[];
    bN.textContent="▶ Add a tree";
    drawMain(); drawStrip(); refresh();
    cap.innerHTML='Press <b>▶ Add a tree</b>. The first tree alone is barely better than guessing &mdash; the power comes from stacking them.';
  }
  bN.addEventListener("click",addTree);
  bA.addEventListener("click",toggleAuto);
  bR.addEventListener("click",reset);
  reset();
})();
</script>
""".replace("__UID__", _uid)

display(HTML(_HTML))


### Your turn

In [ ]:
# 🎯 YOUR TURN — Exercise 3: boosting (trees built one after another).
#
# 💭 Think first: a forest builds its trees independently and averages them. BOOSTING is different —
#    it adds trees in sequence, each one correcting the errors the previous trees still make. Two
#    settings follow directly from that idea:
#      • learning_rate=0.05  -> each new tree takes only a small, safe step (no overcorrecting)
#      • max_depth=5         -> the trees stay shallow; boosting wants many WEAK learners, not deep ones
#
# 🎯 Implement: an XGBClassifier with n_estimators=500, max_depth=5, learning_rate=0.05,
#    subsample=0.8, colsample_bytree=0.8, eval_metric='auc', n_jobs=-1, random_state=RANDOM_STATE.
#    Fit it on X_train, y_train. Store it in `xgb`.
xgb = ...

# already written for you: the scoring
results['XGBoost'] = roc_auc_score(y_test, xgb.predict_proba(X_test)[:, 1])
print(f"XGBoost    ROC-AUC = {results['XGBoost']:.3f}")

# 💭 Think first: LightGBM is a SECOND boosting library — same idea, a different (faster) engine.
#    Do you expect a wildly different score from XGBoost, or roughly the same? Why?
# 🎯 Implement: an LGBMClassifier with the SAME n_estimators, max_depth, learning_rate, subsample
#    and colsample_bytree as the XGBoost above, plus n_jobs=-1, random_state=RANDOM_STATE, verbose=-1.
#    Fit it on X_train, y_train. Store it in `lgbm`.
lgbm = ...

# already written for you: the scoring
results['LightGBM'] = roc_auc_score(y_test, lgbm.predict_proba(X_test)[:, 1])
print(f"LightGBM   ROC-AUC = {results['LightGBM']:.3f}")

Boosting comes out ahead. It can isolate the rare, high risk combinations of sensor
values (for example high temperature together with high vibration and low pressure) that
a forest tends to average away.

### What did the boosted model rely on?

The model is trained &mdash; now let's *read* it. The most useful readout is **feature importance by
gain**: how much each sensor actually improved the predictions. (We use *gain*, not the raw number of
splits &mdash; with 35 pure-noise columns, simply counting how often a feature gets used is misleading.)
We then line it up against the Random Forest's ranking to see where boosting and bagging disagree.

In [ ]:
# Importance by GAIN: how much each feature improved the model (not how often it was split on).
gain = xgb.get_booster().get_score(importance_type='gain')
xgb_imp = pd.Series(gain).reindex(X_train.columns).fillna(0.0)
xgb_imp = (xgb_imp / xgb_imp.sum()).sort_values(ascending=False)

print('Top features XGBoost relied on (share of total gain):')
print((xgb_imp.head(8) * 100).round(1).to_string())

noise_cols = [c for c in X_train.columns if c.startswith('sensor_noise_')]
print(f"\nPer-column noise floor (largest single noise feature): {xgb_imp[noise_cols].max():.1%}"
      f"  ->  every real driver sits well above it.")

# Where do boosting and the forest disagree? Look at the pure-interaction (XOR) phase_* features.
phase = [c for c in X_train.columns if c.startswith('phase_')]
ranks = pd.DataFrame({'XGBoost_rank': xgb_imp.rank(ascending=False).astype(int),
                      'RandomForest_rank': imp.rank(ascending=False).astype(int)})
print('\nInteraction (XOR) features - importance rank in each model (1 = most important):')
print(ranks.loc[phase].to_string())

In [ ]:
#@title 📊 Visualization: what XGBoost relied on, vs the Random Forest (double-click to view the code) { display-mode: "form" }
from IPython.display import HTML, display

def _imp_compare(xgb_imp, rf_imp, k=8):
    def kind(n):
        if n.startswith('sensor_noise_'): return ('noise', '#b8b4c8')
        if n.startswith('phase_'):        return ('interaction (XOR)', '#e0934a')
        base = n.split('_')[0]
        if base in ('temperature','vibration','pressure','rotation','motor','oil','acoustic'):
            return ('true sensor', '#764ba2')
        return ('category', '#4d8fd6')

    def column(imp, title):
        top = imp.sort_values(ascending=False).head(k)
        mx = float(top.iloc[0]) or 1.0
        rows = ''
        for name, val in top.items():
            lab, c = kind(name); wpc = float(val) / mx * 100
            rows += (f'<div class="ic-row"><div class="ic-name" title="{lab}">{name}</div>'
                     f'<div class="ic-track"><div class="ic-bar" style="width:{wpc:.1f}%;background:{c}"></div></div>'
                     f'<div class="ic-val">{float(val)*100:.1f}%</div></div>')
        return f'<div class="ic-col"><div class="ic-ch">{title}</div>{rows}</div>'

    legend = ''.join(f'<span><span class="ic-sw" style="background:{c}"></span>{lab}</span>' for lab, c in
                     [('true sensor','#764ba2'),('interaction (XOR)','#e0934a'),('category','#4d8fd6'),('noise','#b8b4c8')])

    style = '''
    <style>
    .ic{font-family:system-ui,Segoe UI,Roboto,sans-serif;background:linear-gradient(135deg,#f6f8ff,#fbf5ff);
        border:1px solid #ecebff;border-radius:18px;padding:18px;max-width:840px;color:#2c2350}
    .ic-h{font-size:20px;font-weight:800;color:#3b2d6b;margin:0 0 12px}
    .ic-row2{display:flex;gap:14px;flex-wrap:wrap}
    .ic-col{flex:1 1 340px;background:#fff;border:1px solid #e6e8ee;border-radius:14px;padding:12px 14px}
    .ic-ch{font-size:13px;font-weight:800;color:#2c2350;margin-bottom:9px}
    .ic-row{display:flex;align-items:center;gap:8px;margin:5px 0}
    .ic-name{flex:0 0 116px;font-size:10.5px;color:#3b2d6b;font-weight:600;text-align:right;
             white-space:nowrap;overflow:hidden;text-overflow:ellipsis}
    .ic-track{flex:1;height:13px;background:#eef0f5;border-radius:7px;overflow:hidden}
    .ic-bar{height:100%;border-radius:7px}
    .ic-val{flex:0 0 38px;font-size:10.5px;color:#6b6685;font-weight:700;text-align:right}
    .ic-leg{display:flex;gap:14px;flex-wrap:wrap;font-size:11px;color:#6b6685;font-weight:600;margin:11px 2px 0}
    .ic-sw{display:inline-block;width:11px;height:11px;border-radius:3px;vertical-align:middle;margin-right:5px}
    .ic-cap{font-size:11.5px;color:#6b6685;line-height:1.5;margin-top:12px;background:#fff;border:1px solid #ecebff;
            border-radius:11px;padding:9px 11px}.ic-cap b{color:#3b2d6b}
    </style>'''

    html = style + f'''
    <div class="ic">
      <div class="ic-h">🔍 What each model leaned on (top {k} features)</div>
      <div class="ic-row2">{column(xgb_imp, "🚀 XGBoost (by gain)")}{column(rf_imp, "🌲 Random Forest")}</div>
      <div class="ic-leg">{legend}</div>
      <div class="ic-cap">Both models put the <b>true sensors</b> (temperature, vibration, pressure, rotation speed) on
      top and keep every individual <b>noise</b> column near the floor. The tell-tale difference is the orange
      <b>interaction (XOR) features</b>: XGBoost ranks the <code>phase_*</code> signals far higher than the forest does.
      Those features carry almost no information alone &mdash; only <i>as pairs</i> &mdash; so the forest's random per-split
      feature sampling tends to average them away, while boosting's greedy, sequential splits dig them out. Capturing
      those extra interactions is a big part of why XGBoost edges out the Random Forest here.</div>
    </div>'''
    display(HTML(html))

_imp_compare(xgb_imp, imp)


## Exercise 4. Stacking with a meta model

Stacking combines several different models and lets a final meta model learn how much to
trust each one. To avoid leakage, the meta model is trained on out of fold predictions,
that is, predictions made by each base model on data it did not see during training.

We use three base models of different nature: a Support Vector Machine with an RBF kernel,
a Random Forest, and XGBoost. The SVM is a kernel model that captures smooth patterns the
trees struggle with, while the trees capture sharp rules the SVM misses. A simple
`LinearRegression` is used as the meta model so that its weights are easy to read.

**Task.**

1. Evaluate the SVM on its own.
2. Build the stack from out of fold predictions and a linear meta model, then evaluate it.

scikit-learn references:
[StackingClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.StackingClassifier.html),
[stacking guide](https://scikit-learn.org/stable/modules/ensemble.html#stacked-generalization),
[SVC](https://scikit-learn.org/stable/modules/generated/sklearn.svm.SVC.html),
[cross_val_predict](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.cross_val_predict.html),
[LinearRegression](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html).

In [ ]:
#@title 🧩 Visualization: what stacking actually does (double-click to view the code) { display-mode: "form" }
from IPython.display import HTML, display
import uuid as _uuid

_uid = "stk" + _uuid.uuid4().hex[:8]

_HTML = r"""
<style>
#__UID__{font-family:system-ui,Segoe UI,Roboto,sans-serif;background:linear-gradient(135deg,#f6f8ff,#fbf5ff);
  border:1px solid #ecebff;border-radius:18px;padding:18px;max-width:900px;color:#2c2350}
#__UID__ .sk-h{font-size:20px;font-weight:800;color:#3b2d6b;margin:0 0 4px}
#__UID__ .sk-sub{font-size:12.5px;color:#6b6685;line-height:1.55;margin:0 0 14px}
#__UID__ .sk-sub b{color:#3b2d6b}
/* architecture flow */
#__UID__ .sk-arch{background:#fff;border:1px solid #e6e8ee;border-radius:14px;padding:12px;margin-bottom:14px}
#__UID__ .sk-lvl{display:flex;align-items:center;justify-content:center;gap:8px;flex-wrap:wrap}
#__UID__ .sk-node{font-size:11.5px;font-weight:700;color:#3b2d6b;background:#f3f1fb;border:1px solid #e4e0f6;
  border-radius:9px;padding:6px 11px}
#__UID__ .sk-base{display:flex;gap:8px}
#__UID__ .sk-bm{font-size:11px;font-weight:800;color:#fff;border-radius:9px;padding:6px 12px}
#__UID__ .sk-feat{font-size:11.5px;font-weight:700;color:#5b3da8;background:#efeafb;border:1px dashed #c9b8ec;border-radius:9px;padding:6px 12px}
#__UID__ .sk-meta{font-size:11.5px;font-weight:800;color:#fff;background:linear-gradient(135deg,#667eea,#764ba2);border-radius:9px;padding:7px 14px}
#__UID__ .sk-final{font-size:11.5px;font-weight:800;color:#1f7a45;background:#eafaf0;border:1px solid #bfe9cf;border-radius:9px;padding:6px 12px}
#__UID__ .sk-down{color:#b9a9e6;font-size:15px;text-align:center;margin:3px 0}
/* table */
#__UID__ .sk-card{background:#fff;border:1px solid #e6e8ee;border-radius:14px;padding:13px 15px}
#__UID__ .sk-ct{font-size:13px;font-weight:800;color:#2c2350;margin-bottom:2px}
#__UID__ .sk-cs{font-size:11px;color:#6b6685;margin-bottom:10px;line-height:1.4}
#__UID__ .sk-tbl{width:100%;border-collapse:separate;border-spacing:0 4px;font-size:11.5px}
#__UID__ .sk-tbl th{font-size:10px;color:#6b6685;font-weight:700;padding:2px 6px;text-align:center}
#__UID__ .sk-tbl th.lh{text-align:left}
#__UID__ .sk-tbl td{padding:5px 6px;text-align:center;background:#fbfbff;border-top:1px solid #f0f0f6;border-bottom:1px solid #f0f0f6}
#__UID__ .sk-tbl td.mc{font-weight:800;color:#3b2d6b;text-align:left;border-left:1px solid #f0f0f6;border-radius:8px 0 0 8px}
#__UID__ .sk-tbl td.last{border-right:1px solid #f0f0f6;border-radius:0 8px 8px 0}
#__UID__ .sk-tr.dim{opacity:.18}
#__UID__ .sk-tr.cur td{background:#f3eefb}
#__UID__ .sk-cell{display:inline-flex;flex-direction:column;align-items:center;gap:2px;min-width:42px}
#__UID__ .sk-pv{font-weight:700}
#__UID__ .sk-mini{width:38px;height:5px;border-radius:3px;background:#eef0f5;overflow:hidden}
#__UID__ .sk-mini>i{display:block;height:100%}
#__UID__ .sk-ok{color:#1f7a45;font-weight:800}#__UID__ .sk-bad{color:#b5392a;font-weight:800}
#__UID__ .sk-blend{font-weight:800}
#__UID__ .sk-colhint{font-size:9px;color:#a7a3bd;font-weight:600}
/* combination panel */
#__UID__ .sk-combo{background:#faf7ff;border:1px solid #ecebff;border-radius:11px;padding:10px 12px;margin-top:11px;font-size:12px;line-height:1.7;min-height:22px}
#__UID__ .sk-combo b{color:#3b2d6b}
#__UID__ .sk-wsvm{color:#4d8fd6;font-weight:800}#__UID__ .sk-wrf{color:#39b36a;font-weight:800}#__UID__ .sk-wxgb{color:#9a6fe2;font-weight:800}
#__UID__ .sk-ctrl{display:flex;gap:8px;align-items:center;margin-top:12px;flex-wrap:wrap}
#__UID__ .sk-btn{cursor:pointer;border:none;border-radius:10px;padding:9px 13px;font-size:12.5px;font-weight:700;color:#fff;
  background:linear-gradient(135deg,#667eea,#764ba2);box-shadow:0 6px 14px rgba(102,126,234,.32)}
#__UID__ .sk-btn.alt{background:linear-gradient(135deg,#8b86a6,#6b6685);box-shadow:none}
#__UID__ .sk-btn:active{transform:translateY(1px)}#__UID__ .sk-btn:disabled{opacity:.45;cursor:default}
#__UID__ .sk-wbadge{font-size:11px;color:#6b6685;font-weight:600;margin-left:auto}
#__UID__ .sk-note{font-size:10.5px;color:#a7a3bd;margin-top:9px;line-height:1.45}
#__UID__ .sk-note b{color:#6b6685}
#__UID__ .sk-why{display:flex;gap:10px;flex-wrap:wrap;margin-top:14px}
#__UID__ .sk-w{flex:1 1 250px;background:#fff;border:1px solid #ecebff;border-radius:11px;padding:9px 11px;font-size:11px;color:#6b6685;line-height:1.45}
#__UID__ .sk-w b{color:#3b2d6b}
</style>
<div id="__UID__">
  <div class="sk-h">🧩 What stacking actually does</div>
  <div class="sk-sub">Stacking trains <b>several different models</b>, then trains <b>one more model on top of their
  predictions</b>. The trick: each base model's prediction becomes a <b>new feature</b>. Those predictions form a
  small table, and a <b>meta-model</b> learns how much to trust each one &mdash; so it can override a base model when the
  others disagree.</div>

  <div class="sk-arch">
    <div class="sk-lvl"><div class="sk-node">🛠️ machine sensor features (X)</div></div>
    <div class="sk-down">▼ &nbsp; each base model makes its own prediction</div>
    <div class="sk-lvl"><div class="sk-base">
      <div class="sk-bm" style="background:#4d8fd6">🤖 SVM</div>
      <div class="sk-bm" style="background:#39b36a">🌲 RF</div>
      <div class="sk-bm" style="background:#9a6fe2">🚀 XGB</div></div></div>
    <div class="sk-down">▼ &nbsp; their predictions are stacked into a NEW table (predictions = features)</div>
    <div class="sk-lvl"><div class="sk-feat">📋 table of base-model predictions</div></div>
    <div class="sk-down">▼ &nbsp; a second model learns to weight that table</div>
    <div class="sk-lvl"><div class="sk-meta">🧮 meta-model (learns the weights)</div></div>
    <div class="sk-down">▼</div>
    <div class="sk-lvl"><div class="sk-final">⭐ final blended prediction</div></div>
  </div>

  <div class="sk-card">
    <div class="sk-ct">📋 The meta-model's training table: each base model's prediction is one column</div>
    <div class="sk-cs">Rows = machines, columns = "what each base model thinks" (probability of failure). The meta-model reads this table and learns the blend &mdash; press the button to feed in machines one by one.</div>
    <table class="sk-tbl">
      <thead><tr>
        <th class="lh">machine</th>
        <th>🤖 SVM<div class="sk-colhint">feature 1</div></th>
        <th>🌲 RF<div class="sk-colhint">feature 2</div></th>
        <th>🚀 XGB<div class="sk-colhint">feature 3</div></th>
        <th>🧮 meta blend</th>
        <th>actual</th>
      </tr></thead>
      <tbody class="sk-body"></tbody>
    </table>
    <div class="sk-combo">Press <b>▶ Feed next machine</b> to send a machine through the three base models and into the meta-model.</div>
    <div class="sk-ctrl">
      <button class="sk-btn" data-act="next">▶ Feed next machine</button>
      <button class="sk-btn" data-act="auto">⏩ Auto</button>
      <button class="sk-btn alt" data-act="reset">↺</button>
      <span class="sk-wbadge">meta-model trust: <span class="sk-wsvm">SVM 30%</span> · <span class="sk-wrf">RF 15%</span> · <span class="sk-wxgb">XGB 55%</span></span>
    </div>
    <div class="sk-note"><b>Where do the table's numbers come from?</b> To keep them honest, each base model predicts a machine
    only when that machine was <i>held out</i> of its training (out-of-fold) &mdash; otherwise the predictions would look
    unrealistically good and the meta-model would over-trust them.</div>
  </div>

  <div class="sk-why">
    <div class="sk-w">🎭 <b>Why different base models?</b> An SVM draws smooth curved boundaries; the trees cut sharp staircases. They get <i>different</i> machines wrong, so the meta-model can lean on whichever is right (see #C and #D, where RF is wrong but the blend is right).</div>
    <div class="sk-w">⚖️ <b>Why a meta-model, not a plain average?</b> Averaging trusts every model equally. The meta-model <i>learns</i> the weights from the table, so a stronger model counts more and a redundant one can count less (even negatively).</div>
  </div>
</div>
<script>
(function(){
  var root=document.getElementById("__UID__");
  var body=root.querySelector(".sk-body"), combo=root.querySelector(".sk-combo");
  var bN=root.querySelector('[data-act="next"]'), bA=root.querySelector('[data-act="auto"]'), bR=root.querySelector('[data-act="reset"]');
  var W={svm:0.30, rf:0.15, xgb:0.55};
  var M=[
    {id:"#A", y:1, svm:0.80, rf:0.75, xgb:0.85},
    {id:"#B", y:0, svm:0.20, rf:0.30, xgb:0.15},
    {id:"#C", y:1, svm:0.75, rf:0.40, xgb:0.72},
    {id:"#D", y:0, svm:0.35, rf:0.62, xgb:0.30},
    {id:"#E", y:1, svm:0.55, rf:0.60, xgb:0.78}
  ];
  var shown=0, timer=null;
  function blend(m){return W.svm*m.svm + W.rf*m.rf + W.xgb*m.xgb;}
  function riskCol(p){return p>=0.5 ? "#e0796d" : "#39b36a";}
  function cell(p){
    return '<div class="sk-cell"><span class="sk-pv">'+p.toFixed(2)+'</span>'+
           '<span class="sk-mini"><i style="width:'+Math.round(p*100)+'%;background:'+riskCol(p)+'"></i></span></div>';
  }
  function mark(p,y){return (p>=0.5?1:0)===y ? '<span class="sk-ok">✓</span>' : '<span class="sk-bad">✗</span>';}

  function render(){
    body.innerHTML="";
    M.forEach(function(m,i){
      var tr=document.createElement("tr");
      tr.className="sk-tr"+(i>=shown?" dim":"")+(i===shown-1?" cur":"");
      if(i<shown){
        var b=blend(m), bp=(b>=0.5)?"⚠️ FAIL":"✅ healthy";
        tr.innerHTML='<td class="mc">'+m.id+'</td>'+
          '<td>'+cell(m.svm)+' '+mark(m.svm,m.y)+'</td>'+
          '<td>'+cell(m.rf)+' '+mark(m.rf,m.y)+'</td>'+
          '<td>'+cell(m.xgb)+' '+mark(m.xgb,m.y)+'</td>'+
          '<td><span class="sk-blend" style="color:'+riskCol(b)+'">'+b.toFixed(2)+'</span><br><span style="font-size:9.5px;color:#6b6685">'+bp+' '+mark(b,m.y)+'</span></td>'+
          '<td class="last">'+(m.y?'🔴 fail':'🟢 healthy')+'</td>';
      } else {
        tr.innerHTML='<td class="mc">'+m.id+'</td><td>·</td><td>·</td><td>·</td><td>—</td><td class="last">·</td>';
      }
      body.appendChild(tr);
    });
  }
  function describe(m){
    var b=blend(m);
    var parts='<span class="sk-wsvm">0.30·'+m.svm.toFixed(2)+'</span> + '+
              '<span class="sk-wrf">0.15·'+m.rf.toFixed(2)+'</span> + '+
              '<span class="sk-wxgb">0.55·'+m.xgb.toFixed(2)+'</span> = <b>'+b.toFixed(2)+'</b>';
    var verdict=(b>=0.5)===!!m.y ? 'matches the truth ✓' : 'misses ✗';
    var note='';
    var rfWrong=((m.rf>=0.5?1:0)!==m.y), blendRight=((b>=0.5?1:0)===m.y);
    if(rfWrong && blendRight) note=' &nbsp;<b>RF alone was wrong here</b>, but the weighted blend still got it right.';
    combo.innerHTML='<b>Machine '+m.id+':</b> meta-model combines the three predictions &rarr; '+parts+
      ' &rarr; <b>'+(b>=0.5?'FAIL':'healthy')+'</b>, '+verdict+'.'+note;
  }
  function next(){
    if(shown>=M.length) return;
    shown++; render(); describe(M[shown-1]);
    if(shown>=M.length){stopAuto(); bN.disabled=true; bN.textContent="✓ table built";}
  }
  function stopAuto(){if(timer){clearInterval(timer);timer=null;bA.textContent="⏩ Auto";}}
  function toggleAuto(){if(timer){stopAuto();return;}bA.textContent="⏸";timer=setInterval(function(){if(shown>=M.length){stopAuto();return;}next();},1100);}
  function reset(){stopAuto(); shown=0; bN.disabled=false; bN.textContent="▶ Feed next machine"; render();
    combo.innerHTML='Press <b>▶ Feed next machine</b> to send a machine through the three base models and into the meta-model.';}
  bN.addEventListener("click",next); bA.addEventListener("click",toggleAuto); bR.addEventListener("click",reset);
  reset();
})();
</script>
""".replace("__UID__", _uid)

display(HTML(_HTML))


### Your turn. The SVM on its own

In [ ]:
# 🎯 YOUR TURN — a deliberately DIFFERENT base model for the stack.
#
# 💭 Think first: every model so far has been tree-based. The SVM (with an RBF kernel) is a kernel
#    model — it draws smooth, curved boundaries that trees can only approximate with staircases.
#    For stacking we WANT a model that makes different mistakes than the trees. Why does that matter?
#    (Hint: a blend only helps when the models disagree in useful ways.)
#    The SVM needs scaled + feature-selected inputs, already prepared for you as X_train_svm.
#
# 🎯 Implement: an SVC with probability=True and random_state=RANDOM_STATE, fit on
#    X_train_svm, y_train. Store it in `svm`.
svm = ...

# already written for you: the scoring (note the SVM uses X_test_svm)
results['SVM'] = roc_auc_score(y_test, svm.predict_proba(X_test_svm)[:, 1])
print(f"SVM        ROC-AUC = {results['SVM']:.3f}")

### Your turn. The stack

**What you'll build — and the tools for it**

*Stratified k-fold cross-validation.* Split the training machines into *k* equal folds, each keeping
the same failure rate (that is the **stratified** part — it matters when we care about both classes).
Then, *k* times, train on *k − 1* folds and predict the held-out fold — so every training machine
gets a prediction from a model that **never saw it**. Those are the **out-of-fold (OOF)** predictions,
and they are honest features for the meta-model.

- `cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)` — already built for you.
- `cross_val_predict(model, X, y_train, cv=cv, method='predict_proba')[:, 1]` — runs that whole
  *train-on-the-rest / predict-the-held-out-fold* loop for you and returns the **failure-probability
  column** for every training row.

Mind the inputs: the SVM was prepared on `X_train_svm`, the trees on `X_train`.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

# 🎯 YOUR TURN — Exercise 4: build the stack. This is the conceptual heart of the lab.
#    (See the note just above for how stratified cross-validation and cross_val_predict work.)
#
# 🎯 Step 1 — out-of-fold predictions on TRAIN: one probability column per base model (svm, rf, xgb).
#    💭 How: for each base model call
#            cross_val_predict(model, X, y_train, cv=cv, method='predict_proba')[:, 1]
#       Mind the inputs:  SVM -> X_train_svm ;  RF -> X_train ;  XGB -> X_train.
oof_svm = ...
oof_rf  = ...
oof_xgb = ...
oof = np.column_stack([oof_svm, oof_rf, oof_xgb])

# 🎯 Step 2 — build the meta-model's TEST features (call it `test_meta`).
#    The three base models (svm, rf, xgb) are ALREADY fitted on the full training set, so for the
#    test set you do NOT need cross-validation — just ask each model for its failure probability and
#    place the three columns side by side.
#    💭 How: get one column per model with   model.predict_proba(<its test input>)[:, 1]
#            then combine the three columns with   np.column_stack([...])
#            Inputs:  SVM -> X_test_svm ;  RF -> X_test ;  XGB -> X_test.
#    💭 Think: why is a plain prediction OK here, but would have been cheating in Step 1?
test_meta = ...

# 🎯 Step 3 — fit a LinearRegression() meta-model on (oof, y_train), then predict on test_meta to
#    get `stack_proba`. (A linear meta-model keeps the learned weights easy to read in the next cell.)
meta = ...
stack_proba = ...

# already written for you: the scoring
results['Stacking'] = roc_auc_score(y_test, stack_proba)
print(f"Stacking   ROC-AUC = {results['Stacking']:.3f}")

In [ ]:
#@title 📊 Visualization: how the meta-model blends the base models (double-click to view the code) { display-mode: "form" }
from IPython.display import HTML, display

def _stack_report(weights, base_auc, corr, stack_auc):
    names = ['SVM', 'RandomForest', 'XGBoost']
    wmax = max(abs(w) for w in weights.values()) or 1.0
    best_base = max(base_auc, key=base_auc.get)

    def wbar(w):                                   # signed weight bar around a centre line
        pct = abs(w) / wmax * 50
        if w >= 0:
            return (f'<div class="sr-wt"><div class="sr-half"></div><div class="sr-half">'
                    f'<div class="sr-pos" style="width:{pct:.0f}%"></div></div></div>')
        return (f'<div class="sr-wt"><div class="sr-half" style="display:flex;justify-content:flex-end">'
                f'<div class="sr-neg" style="width:{pct:.0f}%"></div></div><div class="sr-half"></div></div>')

    rows = ''
    for n in names:
        w = weights[n]; sign = '+' if w >= 0 else '−'
        wc = '#764ba2' if w >= 0 else '#e0796d'
        rows += (f'<div class="sr-row"><div class="sr-nm">{n}</div>'
                 f'<div class="sr-auc">AUC {base_auc[n]:.3f}</div>{wbar(w)}'
                 f'<div class="sr-wv" style="color:{wc}">{sign}{abs(w):.2f}</div></div>')

    # pairwise correlations among the out-of-fold predictions
    pairs = [('SVM','RF',corr[0][1]), ('SVM','XGB',corr[0][2]), ('RF','XGB',corr[1][2])]
    least = min(pairs, key=lambda p: p[2])
    chips = ''.join(f'<span class="sr-chip">{a}&ndash;{b}: <b>{c:.2f}</b></span>' for a,b,c in pairs)
    delta = stack_auc - base_auc[best_base]

    style = '''
    <style>
    .sr{font-family:system-ui,Segoe UI,Roboto,sans-serif;background:linear-gradient(135deg,#f6f8ff,#fbf5ff);
        border:1px solid #ecebff;border-radius:18px;padding:18px;max-width:780px;color:#2c2350}
    .sr-h{font-size:20px;font-weight:800;color:#3b2d6b;margin:0 0 4px}
    .sr-s{font-size:11.5px;color:#6b6685;margin:0 0 12px;line-height:1.45}
    .sr-card{background:#fff;border:1px solid #e6e8ee;border-radius:14px;padding:12px 14px;margin-bottom:11px}
    .sr-ct{font-size:12.5px;font-weight:800;color:#2c2350;margin-bottom:8px}
    .sr-row{display:flex;align-items:center;gap:9px;margin:7px 0}
    .sr-nm{flex:0 0 96px;font-size:11.5px;font-weight:700;color:#3b2d6b}
    .sr-auc{flex:0 0 70px;font-size:11px;color:#6b6685;font-weight:600}
    .sr-wt{flex:1;display:flex;height:16px;background:#f4f4fa;border-radius:6px;overflow:hidden;border:1px solid #eceaf5}
    .sr-half{width:50%;display:flex;align-items:center}
    .sr-half:first-child{border-right:1px dashed #d8d5ea}
    .sr-pos{height:100%;background:linear-gradient(90deg,#8a6cc0,#764ba2);border-radius:0 4px 4px 0}
    .sr-neg{height:100%;background:linear-gradient(90deg,#e0796d,#d4604f);border-radius:4px 0 0 4px}
    .sr-wv{flex:0 0 42px;text-align:right;font-size:12px;font-weight:800}
    .sr-hint{font-size:10.5px;color:#a7a3bd;text-align:center;margin-top:3px}
    .sr-chips{display:flex;gap:8px;flex-wrap:wrap}
    .sr-chip{font-size:11px;color:#6b6685;background:#f3eefb;border:1px solid #e4d9f7;border-radius:8px;padding:3px 9px}
    .sr-chip b{color:#5b3da8}
    .sr-res{display:flex;align-items:baseline;gap:10px;flex-wrap:wrap}
    .sr-big{font-size:26px;font-weight:800;color:#1f7a45}
    .sr-delta{font-size:12px;font-weight:800;color:#1f7a45;background:#eafaf0;border-radius:7px;padding:2px 8px}
    .sr-cap{font-size:11.5px;color:#6b6685;line-height:1.5;margin-top:11px;background:#fff;border:1px solid #ecebff;border-radius:11px;padding:9px 11px}
    .sr-cap b{color:#3b2d6b}
    </style>'''

    html = style + f'''
    <div class="sr">
      <div class="sr-h">📊 Inside the blend</div>
      <div class="sr-s">Each base model's standalone skill (AUC) and the weight the linear meta-model gave it. The
      dashed line is zero &mdash; bars to the right are positive weights, to the left negative.</div>

      <div class="sr-card">
        <div class="sr-ct">⚖️ Meta-model weights (how each base model enters the blend)</div>
        {rows}
        <div class="sr-hint">&larr; negative (correction)&nbsp;&nbsp;|&nbsp;&nbsp;positive (trusted) &rarr;</div>
      </div>

      <div class="sr-card">
        <div class="sr-ct">🎭 How differently do the base models predict? (correlation of their out-of-fold scores)</div>
        <div class="sr-chips">{chips}</div>
        <div class="sr-hint" style="text-align:left;margin-top:7px">Lower = more diverse. The least-correlated pair is
        <b>{least[0]}&ndash;{least[1]}</b> &mdash; that diversity is what the meta-model cashes in.</div>
      </div>

      <div class="sr-card">
        <div class="sr-ct">🏆 Result</div>
        <div class="sr-res"><span class="sr-big">{stack_auc:.3f}</span>
          <span>stacking AUC</span>
          <span class="sr-delta">+{delta:.3f} vs best base ({best_base} {base_auc[best_base]:.3f})</span></div>
      </div>

      <div class="sr-cap">Read the weights as a <b>recipe, not a ranking</b>. XGBoost is trusted most; the SVM gets a
      real positive weight because it is the most <i>different</i> model and adds what the trees miss; a base can even get a
      <b>negative</b> weight (used as a correction when it overlaps a stronger model) without being &ldquo;bad&rdquo;. The blend
      clears the best single model precisely because the bases disagree in useful ways.</div>
    </div>'''
    display(HTML(html))

import numpy as _np
_stack_report(
    dict(zip(['SVM','RandomForest','XGBoost'], meta.coef_)),
    {k: results[k] for k in ['SVM','RandomForest','XGBoost']},
    _np.corrcoef(oof, rowvar=False),
    results['Stacking'],
)


### Reading the result

Two things here are worth really understanding.

**The blend beats every base model.** On their own the bases score roughly SVM ≈ 0.89, Random Forest
≈ 0.87, XGBoost ≈ 0.90; the stack lands around **0.91**, above the best single one. The gain is small
but free &mdash; it costs only compute.

**The meta-weights are *combination coefficients*, not report cards.** XGBoost (strongest and most
informative) gets the largest positive weight, and the SVM earns a solid positive weight too &mdash; as a
kernel model it makes *different* mistakes than the trees and adds information they lack. The Random
Forest can even get a **negative** weight, and that does **not** mean it is useless: it is highly
correlated with XGBoost (both are tree ensembles) but a little weaker, so the linear meta-model keeps it
around as a small *correction term* rather than a standalone vote. Negative weights are normal in
stacking &mdash; the meta-model only optimizes the best *blend*, it does not rank the parts.

**Diversity is what makes stacking pay.** The lift is proportional to how *differently* the base models
err. Here the SVM is the odd one out (a smooth kernel boundary vs the trees' staircases), so it supplies
the complementary signal that pushes the blend past pure boosting.

### ⭐ Bonus. Dynamic weights: let the blend change per machine (FWLS)

Our stack used **one fixed set of weights** for every machine. But this dataset has different
*regions*: a smooth, curved risk zone where the kernel SVM shines, and sharp high-risk "pockets"
that the trees carve out better. A single weighting can't be best in both.

**Feature-weighted linear stacking** fixes this: we let each base model's weight **depend on the
machine's features**. The trick is simple &mdash; instead of feeding the meta-model only the three
predictions, we also feed each prediction *multiplied by* a few key sensors. With a linear
meta-model, the effective weight on each base model then becomes a linear function of those
sensors:

$$\text{weight}_k(x) = a_k + b_k \cdot \text{sensors}(x)$$

so the blend can trust the SVM in one region and XGBoost in another, instead of compromising.

*(This is the idea behind Feature-Weighted Linear Stacking, Sill et al., from the Netflix Prize.)*

In [ ]:
#@title 🗺️ Schema: what changes vs plain stacking (double-click to view the code) { display-mode: "form" }
from IPython.display import HTML, display

display(HTML(r'''
<style>
.ar{font-family:system-ui,Segoe UI,Roboto,sans-serif;background:linear-gradient(135deg,#f6f8ff,#fbf5ff);
    border:1px solid #ecebff;border-radius:18px;padding:18px;max-width:880px;color:#2c2350}
.ar-h{font-size:19px;font-weight:800;color:#3b2d6b;margin:0 0 3px}
.ar-s{font-size:12px;color:#6b6685;margin:0 0 14px;line-height:1.45}
.ar-flow{background:#fff;border:1px solid #e6e8ee;border-radius:14px;padding:12px 14px;margin-bottom:12px}
.ar-flow.new{border:1.5px solid #c9b8ec;box-shadow:0 5px 16px rgba(118,75,162,.12)}
.ar-tag{display:inline-block;font-size:10.5px;font-weight:800;border-radius:7px;padding:3px 9px;margin-bottom:10px;
    background:#eef0f5;color:#6b6685}
.ar-tag.tagnew{background:linear-gradient(135deg,#667eea,#764ba2);color:#fff}
.ar-line{display:flex;align-items:center;gap:6px;flex-wrap:wrap}
.ar-node{font-size:11px;font-weight:700;color:#3b2d6b;background:#f6f5fc;border:1px solid #e9e6f5;border-radius:10px;
    padding:8px 10px;text-align:center;line-height:1.25}
.ar-nl{font-size:9px;font-weight:600;color:#a7a3bd;margin-top:2px}
.ar-base{background:#eef7f1;border-color:#cfe9da}
.ar-meta{background:linear-gradient(135deg,#eef1ff,#f3eefb);border-color:#d9d3f3;color:#3b2d6b}
.ar-out{background:#eafaf0;border-color:#bfe9cf;color:#1f7a45}
.ar-arr{color:#b9a9e6;font-size:16px;font-weight:800}
.ar-extra{display:flex;align-items:flex-start;gap:8px;margin-top:11px;background:#fdf4ea;border:1px dashed #e6b88a;
    border-radius:10px;padding:9px 11px;font-size:11px;color:#8a5a28;line-height:1.45}
.ar-extra b{color:#7a4a18}
.ar-wire{display:flex;align-items:center;gap:6px;margin-top:8px;font-size:10px;color:#c07a2a;font-weight:700}
.ar-wire .ar-dash{flex:1;height:0;border-top:2px dashed #e0934a}
</style>
<div class="ar">
  <div class="ar-h">🗺️ What changes here: the meta-model gets to see the sensors too</div>
  <div class="ar-s">Plain stacking blends the three predictions with <b>one fixed recipe</b>. Dynamic stacking adds a
  single new wire &mdash; it also feeds the <b>original sensors</b> into the meta-model &mdash; so the blend weights can
  change from machine to machine.</div>

  <div class="ar-flow">
    <div class="ar-tag">Plain stacking (what we just did)</div>
    <div class="ar-line">
      <div class="ar-node">🛠️ sensors<div class="ar-nl">X</div></div><span class="ar-arr">&rarr;</span>
      <div class="ar-node ar-base">🤖 🌲 🚀<div class="ar-nl">base models</div></div><span class="ar-arr">&rarr;</span>
      <div class="ar-node">📋 predictions</div><span class="ar-arr">&rarr;</span>
      <div class="ar-node ar-meta">🧮 meta-model<div class="ar-nl">one fixed weighting</div></div><span class="ar-arr">&rarr;</span>
      <div class="ar-node ar-out">⭐ blend</div>
    </div>
  </div>

  <div class="ar-flow new">
    <div class="ar-tag tagnew">Dynamic stacking (FWLS) — what we do now</div>
    <div class="ar-line">
      <div class="ar-node">🛠️ sensors<div class="ar-nl">X</div></div><span class="ar-arr">&rarr;</span>
      <div class="ar-node ar-base">🤖 🌲 🚀<div class="ar-nl">base models</div></div><span class="ar-arr">&rarr;</span>
      <div class="ar-node">📋 predictions</div><span class="ar-arr">&rarr;</span>
      <div class="ar-node ar-meta">🧮 meta-model<div class="ar-nl">weights depend on the sensors</div></div><span class="ar-arr">&rarr;</span>
      <div class="ar-node ar-out">⭐ blend<div class="ar-nl">per-machine weights</div></div>
    </div>
    <div class="ar-wire">🛠️ sensors <span class="ar-dash"></span> 🆕 also wired straight into &rarr; 🧮</div>
    <div class="ar-extra">🆕 <b>The one new wire.</b> The same sensors that fed the base models are now <b>also fed into the
    meta-model</b>. So instead of one weight per base model, the weight becomes a little formula in the sensors &mdash;
    letting the blend trust the SVM in one region and XGBoost in another.</div>
  </div>
</div>
'''))

In [ ]:
# Feature-Weighted Linear Stacking: make the blend weights depend on the machine.
# A few sensors that define the "operating region" (smooth zone vs sharp risk pockets).
gate_features = ['temperature', 'vibration', 'pressure', 'rotation_speed']
gate_scaler = StandardScaler().fit(X_train[gate_features])
Z_train = gate_scaler.transform(X_train[gate_features])
Z_test  = gate_scaler.transform(X_test[gate_features])

# Augment the meta-features: each base prediction, PLUS that prediction times each gate sensor.
# A linear meta-model on these terms makes each base model's weight a linear function of the
# sensors -> weights that change from machine to machine.
def augment(P, Z):
    return np.column_stack([P] + [P[:, [k]] * Z for k in range(P.shape[1])])

dyn_meta  = LinearRegression().fit(augment(oof, Z_train), y_train)
dyn_proba = dyn_meta.predict(augment(test_meta, Z_test))
results['Stacking (dynamic)'] = roc_auc_score(y_test, dyn_proba)

print(f"Static stack    ROC-AUC = {results['Stacking']:.4f}   (one weighting everywhere)")
print(f"Dynamic stack   ROC-AUC = {results['Stacking (dynamic)']:.4f}   (weights depend on the machine)")

# Recover the EFFECTIVE weight on each base model per machine: weight_k(x) = a_k + b_k . sensors.
base_w  = dyn_meta.coef_[:3]
inter_w = dyn_meta.coef_[3:].reshape(3, len(gate_features))
eff_w   = np.column_stack([base_w[k] + Z_test @ inter_w[k] for k in range(3)])

# Average those weights inside two opposite operating regions.
smooth = (X_test['temperature'] < 60).to_numpy()
sharp  = ((X_test['temperature'] > 75) & (X_test['vibration'] > 4)).to_numpy()
region_weights = {
    '🌫️ smooth region (cool)':      dict(zip(['SVM', 'RF', 'XGB'], eff_w[smooth].mean(0))),
    '🔥 sharp region (hot & shaky)': dict(zip(['SVM', 'RF', 'XGB'], eff_w[sharp].mean(0))),
}
print('\nAverage weight the meta-model puts on each base model, by region:')
for r, w in region_weights.items():
    print(f"  {r:26s} " + "   ".join(f"{n} {v:+.2f}" for n, v in w.items()))

In [ ]:
#@title 📊 Visualization: weights that change with the machine (double-click to view the code) { display-mode: "form" }
from IPython.display import HTML, display

def _dyn_stack_report(region_weights, static_auc, dyn_auc):
    names = ['SVM', 'RF', 'XGB']
    colors = {'SVM': '#4d8fd6', 'RF': '#39b36a', 'XGB': '#9a6fe2'}
    wmax = max(abs(v) for w in region_weights.values() for v in w.values()) or 1.0

    def bar(w, c):
        pct = abs(w) / wmax * 50
        if w >= 0:
            return (f'<div class="dy-wt"><div class="dy-half"></div>'
                    f'<div class="dy-half"><div class="dy-pos" style="width:{pct:.0f}%;background:{c}"></div></div></div>')
        return (f'<div class="dy-wt"><div class="dy-half" style="display:flex;justify-content:flex-end">'
                f'<div class="dy-neg" style="width:{pct:.0f}%"></div></div><div class="dy-half"></div></div>')

    def card(label, w):
        rows = ''
        for n in names:
            v = float(w[n]); sign = '+' if v >= 0 else '−'
            rows += (f'<div class="dy-row"><div class="dy-nm" style="color:{colors[n]}">{n}</div>'
                     f'{bar(v, colors[n])}<div class="dy-wv">{sign}{abs(v):.2f}</div></div>')
        return f'<div class="dy-card"><div class="dy-ct">{label}</div>{rows}<div class="dy-hint">&larr; trusted less&nbsp;&nbsp;|&nbsp;&nbsp;trusted more &rarr;</div></div>'

    (sl, sw), (hl, hw) = list(region_weights.items())
    delta = dyn_auc - static_auc

    style = '''
    <style>
    .dy{font-family:system-ui,Segoe UI,Roboto,sans-serif;background:linear-gradient(135deg,#f6f8ff,#fbf5ff);
        border:1px solid #ecebff;border-radius:18px;padding:18px;max-width:820px;color:#2c2350}
    .dy-h{font-size:20px;font-weight:800;color:#3b2d6b;margin:0 0 3px}
    .dy-s{font-size:11.5px;color:#6b6685;margin:0 0 13px;line-height:1.45}
    .dy-row2{display:flex;gap:13px;flex-wrap:wrap}
    .dy-card{flex:1 1 320px;background:#fff;border:1px solid #e6e8ee;border-radius:14px;padding:12px 15px}
    .dy-ct{font-size:13px;font-weight:800;color:#2c2350;margin-bottom:9px}
    .dy-row{display:flex;align-items:center;gap:9px;margin:7px 0}
    .dy-nm{flex:0 0 42px;font-size:12px;font-weight:800}
    .dy-wt{flex:1;display:flex;height:16px;background:#f4f4fa;border-radius:6px;overflow:hidden;border:1px solid #eceaf5}
    .dy-half{width:50%;display:flex;align-items:center}
    .dy-half:first-child{border-right:1px dashed #d8d5ea}
    .dy-pos{height:100%;border-radius:0 4px 4px 0}
    .dy-neg{height:100%;background:linear-gradient(90deg,#e0796d,#d4604f);border-radius:4px 0 0 4px}
    .dy-wv{flex:0 0 42px;text-align:right;font-size:12px;font-weight:800;color:#3b2d6b}
    .dy-hint{font-size:9.5px;color:#a7a3bd;text-align:center;margin-top:6px}
    .dy-auc{display:flex;align-items:center;gap:12px;flex-wrap:wrap;background:#fff;border:1px solid #e6e8ee;
            border-radius:14px;padding:11px 15px;margin-top:12px}
    .dy-aitem{font-size:12px;color:#6b6685}.dy-aitem b{font-size:17px;color:#3b2d6b}
    .dy-arrow{color:#b9a9e6;font-weight:800}
    .dy-delta{font-size:12px;font-weight:800;color:#1f7a45;background:#eafaf0;border-radius:7px;padding:2px 9px}
    .dy-cap{font-size:11.5px;color:#6b6685;line-height:1.5;margin-top:12px;background:#fff;border:1px solid #ecebff;
            border-radius:11px;padding:9px 11px}.dy-cap b{color:#3b2d6b}
    </style>'''

    html = style + f'''
    <div class="dy">
      <div class="dy-h">🎛️ Same models, different trust per region</div>
      <div class="dy-s">The effective weight the dynamic meta-model places on each base model, averaged inside two
      opposite operating regions. Dashed line = zero; bars right = positive, left = negative.</div>
      <div class="dy-row2">{card(sl, sw)}{card(hl, hw)}</div>
      <div class="dy-auc">
        <span class="dy-aitem">static stack <b>{static_auc:.3f}</b></span>
        <span class="dy-arrow">&rarr;</span>
        <span class="dy-aitem">dynamic stack <b>{dyn_auc:.3f}</b></span>
        <span class="dy-delta">+{delta:.3f} AUC</span>
      </div>
      <div class="dy-cap">Look how the trust <b>shifts</b>: in the <b>{sl}</b> the blend leans on the
      <b>SVM</b> ({sw['SVM']:+.2f}), but in the <b>{hl}</b> it swings to <b>XGBoost</b> ({hw['XGB']:+.2f}) and pulls the
      SVM down ({hw['SVM']:+.2f}). That is exactly how the data was built &mdash; kernels win the smooth zone, trees win
      the sharp pockets &mdash; and a single fixed weighting could never capture both. (The Random Forest stays slightly
      negative throughout: still a correction term, as before.)</div>
    </div>'''
    display(HTML(html))

_dyn_stack_report(region_weights, results['Stacking'], results['Stacking (dynamic)'])

## Final comparison

In [ ]:
ranking = pd.Series(results, name='ROC_AUC').sort_values()
display(ranking.round(3).to_frame())

In [ ]:
#@title Plot model ranking { display-mode: "form" }
ax = ranking.plot.barh(figsize=(7,3.5), color='steelblue')
ax.set_xlim(0.75, 0.93); ax.set_xlabel('ROC AUC'); ax.set_title('Model ranking on the test set')
plt.tight_layout(); plt.show()

## Summary

1. Every ensemble beats the single decision tree.
2. A Random Forest reduces variance by bagging; boosting also reduces bias and gives the
   strongest single model here.
3. Feature importance lets us drop most columns with almost no loss in performance.
4. Stacking blends complementary models and finishes on top, with weights that are easy
   to interpret.
5. Letting those weights **depend on the machine** (dynamic / feature-weighted stacking) trusts
   different base models in different regions &mdash; matching how the data was built, and edging a
   little higher still.

## ⭐ Bonus 2. When does dynamic stacking *really* matter?

Quick recap: on the maintenance data, letting the weights depend on the machine barely helped
(**0.911 → 0.912**). Why so little? Because **one model &mdash; XGBoost &mdash; was the best almost
everywhere**. When the same model wins in every corner of the data, there is nothing to *route*:
the smartest thing a gate can do is re-discover "trust XGBoost," which the static stack already did.

**The principle.** Feature-aware stacking only earns its keep when **which model is best changes
from one input to another**. Think of a clinic: if one doctor were the best diagnostician for
*every* patient, you would just always send patients to her (that is a static blend). But if one
doctor is best with hearts and another with lungs, you want a **receptionist who reads each
patient's symptoms and sends them to the right specialist** (that is a dynamic blend). The
receptionist is the meta-model; the symptoms it reads are the features.

**A clean example.** To show this without the maintenance data getting in the way, we build a small
**rigged** dataset (its AUCs below are their own story &mdash; *not* a comparison with the 0.911 above).
Every machine carries a visible **region** flag, `0` or `1`, and we train two **specialists**:

- **Specialist A** sees only the block of sensors that matters in **region 0** &rarr; it is excellent in
  region 0 and barely better than guessing in region 1.
- **Specialist B** sees only the block that matters in **region 1** &rarr; the mirror image: great in
  region 1, useless in region 0.

So *neither is good on its own overall*, and the best one **flips** depending on the region. The
question: can a blend that is allowed to *look at the region flag* beat a blend that must use the
same weights everywhere? The schema shows the two strategies; then we measure them.

In [ ]:
#@title 🗺️ Schema: static blend vs a gate that routes by region (double-click to view the code) { display-mode: "form" }
from IPython.display import HTML, display

display(HTML(r'''
<style>
.rs{font-family:system-ui,Segoe UI,Roboto,sans-serif;background:linear-gradient(135deg,#f6f8ff,#fbf5ff);
    border:1px solid #ecebff;border-radius:18px;padding:18px;max-width:860px;color:#2c2350}
.rs-h{font-size:19px;font-weight:800;color:#3b2d6b;margin:0 0 12px}
.rs-row{display:flex;gap:13px;flex-wrap:wrap}
.rs-card{flex:1 1 360px;background:#fff;border:1px solid #e6e8ee;border-radius:14px;padding:13px 15px}
.rs-ct{font-size:12.5px;font-weight:800;color:#2c2350;margin-bottom:12px}
.rs-flow{display:flex;align-items:center;gap:9px}
.rs-mcol,.rs-outcol{display:flex;flex-direction:column;gap:10px}
.rs-machine{font-size:11px;font-weight:700;color:#3b2d6b;background:#f3f1fb;border:1px solid #e4e0f6;border-radius:9px;
    padding:7px 9px;white-space:nowrap}
.rs-arrow{color:#b9a9e6;font-size:17px;font-weight:800}
.rs-mix{font-size:12px;font-weight:800;color:#6b6685;background:#eef0f5;border:1px solid #dfe2ec;border-radius:11px;
    padding:11px 13px;text-align:center;line-height:1.3}
.rs-sub{font-size:9.5px;font-weight:600;color:#a7a3bd;margin-top:3px}
.rs-out{font-size:11px;font-weight:800;border-radius:9px;padding:7px 9px;text-align:center;white-space:nowrap}
.rs-out.bad{background:#fdecea;color:#b5392a;border:1px solid #f2c4bd}
.rs-out.good{background:#eafaf0;color:#1f7a45;border:1px solid #bfe9cf}
.rs-flow2{display:flex;flex-direction:column;gap:11px}
.rs-line{display:flex;align-items:center;gap:7px;flex-wrap:nowrap}
.rs-gate{font-size:11px;font-weight:800;color:#fff;background:linear-gradient(135deg,#667eea,#764ba2);
    border-radius:9px;padding:7px 9px;white-space:nowrap}
.rs-pick{font-size:11px;font-weight:800;border-radius:9px;padding:7px 9px;white-space:nowrap;color:#fff}
.rs-note{font-size:10.5px;color:#8b86a6;margin-top:12px;line-height:1.45}
.rs-key{font-size:11px;color:#6b6685;line-height:1.5;margin-top:13px;background:#fff;border:1px solid #ecebff;
    border-radius:11px;padding:9px 11px}
.rs-key b{font-weight:800}
</style>
<div class="rs">
  <div class="rs-h">🗺️ Two ways to combine the same two specialists</div>
  <div class="rs-row">

    <div class="rs-card">
      <div class="rs-ct">🔒 Static blend &mdash; one fixed recipe for every machine</div>
      <div class="rs-flow">
        <div class="rs-mcol">
          <div class="rs-machine">❄️ region-0 machine</div>
          <div class="rs-machine">🔥 region-1 machine</div>
        </div>
        <div class="rs-arrow">&rarr;</div>
        <div class="rs-mix">½·A + ½·B<div class="rs-sub">same weights<br>for everyone</div></div>
        <div class="rs-arrow">&rarr;</div>
        <div class="rs-outcol">
          <div class="rs-out bad">≈ half right</div>
          <div class="rs-out bad">≈ half right</div>
        </div>
      </div>
      <div class="rs-note">It can't tell the regions apart, so in each region it averages the <b>expert</b> with the
      <b>coin-flip</b> &mdash; landing in the mediocre middle both times.</div>
    </div>

    <div class="rs-card">
      <div class="rs-ct">🚦 Dynamic blend &mdash; a gate reads the region and routes</div>
      <div class="rs-flow2">
        <div class="rs-line">
          <div class="rs-machine">❄️ region-0</div><div class="rs-arrow">&rarr;</div>
          <div class="rs-gate">🚦 gate</div><div class="rs-arrow">&rarr;</div>
          <div class="rs-pick" style="background:#4d8fd6">trust A</div><div class="rs-arrow">&rarr;</div>
          <div class="rs-out good">right ✅</div>
        </div>
        <div class="rs-line">
          <div class="rs-machine">🔥 region-1</div><div class="rs-arrow">&rarr;</div>
          <div class="rs-gate">🚦 gate</div><div class="rs-arrow">&rarr;</div>
          <div class="rs-pick" style="background:#9a6fe2">trust B</div><div class="rs-arrow">&rarr;</div>
          <div class="rs-out good">right ✅</div>
        </div>
      </div>
      <div class="rs-note">The gate looks at the region flag and hands each machine to the specialist who is the
      <b>expert there</b> &mdash; so it is right in both regions.</div>
    </div>

  </div>
  <div class="rs-key">🤖 <b style="color:#4d8fd6">Specialist A</b> = expert in region 0 &nbsp;·&nbsp;
  <b style="color:#9a6fe2">Specialist B</b> = expert in region 1. &nbsp;The &ldquo;gate&rdquo; isn't a new model &mdash; it is
  just the meta-model <b>letting its weights depend on the region feature</b> (weight on A and B becomes a function of
  the input, instead of one fixed number).</div>
</div>
'''))

In [ ]:
import numpy as np
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_predict
from sklearn.metrics import roc_auc_score

# --- A rigged two-regime dataset (separate from the maintenance data) ------------
# Each machine belongs to region 0 or 1 (an observable flag). In region 0 the label is
# driven by sensor block A; in region 1 by sensor block B. So a model that only knows one
# block is an expert in its own region and blind in the other.
rng = np.random.default_rng(0)
n = 4000
region = rng.integers(0, 2, n)
A = rng.normal(0, 1, (n, 3))          # "block A" sensors
B = rng.normal(0, 1, (n, 3))          # "block B" sensors
score = np.where(region == 0,
                 1.6*A[:, 0] + 1.2*A[:, 1] - 1.0*A[:, 2],
                 1.6*B[:, 0] + 1.2*B[:, 1] - 1.0*B[:, 2])
score += rng.normal(0, 0.7, n)
y = (score > np.median(score)).astype(int)

Xtr_A, Xte_A, Xtr_B, Xte_B, reg_tr, reg_te, ytr, yte = train_test_split(
    A, B, region, y, test_size=0.25, stratify=y, random_state=0)

# --- Two specialists: each sees only its own sensor block ------------------------
specA = LogisticRegression(max_iter=500).fit(Xtr_A, ytr)
specB = LogisticRegression(max_iter=500).fit(Xtr_B, ytr)
cv = StratifiedKFold(5, shuffle=True, random_state=0)
oof = np.column_stack([cross_val_predict(specA, Xtr_A, ytr, cv=cv, method='predict_proba')[:, 1],
                       cross_val_predict(specB, Xtr_B, ytr, cv=cv, method='predict_proba')[:, 1]])
P = np.column_stack([specA.predict_proba(Xte_A)[:, 1], specB.predict_proba(Xte_B)[:, 1]])

def auc(p, mask=None):
    return roc_auc_score(yte[mask] if mask is not None else yte,
                         p[mask] if mask is not None else p)

r0, r1 = (reg_te == 0), (reg_te == 1)
spec_auc = {'Specialist A': [auc(P[:, 0], r0), auc(P[:, 0], r1)],
            'Specialist B': [auc(P[:, 1], r0), auc(P[:, 1], r1)]}
print('Each specialist is an expert in ONE region and a coin-flip in the other:')
for name, (a0, a1) in spec_auc.items():
    print(f'  {name}:  region 0 AUC = {a0:.3f}   region 1 AUC = {a1:.3f}')

# --- Static vs dynamic stack ----------------------------------------------------
static_meta = LinearRegression().fit(oof, ytr)
static_auc = auc(static_meta.predict(P))

g_tr = reg_tr.reshape(-1, 1).astype(float)        # the region flag is the gate feature
g_te = reg_te.reshape(-1, 1).astype(float)
def augment(Pm, G):
    return np.column_stack([Pm] + [Pm[:, [k]] * G for k in range(Pm.shape[1])])
dyn_meta = LinearRegression().fit(augment(oof, g_tr), ytr)
dynamic_auc = auc(dyn_meta.predict(augment(P, g_te)))

print(f'\nStatic stack  ROC-AUC = {static_auc:.3f}   (one compromise weighting)')
print(f'Dynamic stack ROC-AUC = {dynamic_auc:.3f}   (routes by region)')
print(f'Lift = +{dynamic_auc - static_auc:.3f}  AUC  <- huge, because the best model flips by region')

# --- The weights the dynamic blend assigns: weight_k(region) = base_k + slope_k * region ---
wA, wB, wA_reg, wB_reg = dyn_meta.coef_
route_weights = {'region 0': {'Specialist A': wA,          'Specialist B': wB},
                 'region 1': {'Specialist A': wA + wA_reg,  'Specialist B': wB + wB_reg}}
print('\nWeight the DYNAMIC blend puts on each specialist, by region (this IS the routing):')
for reg, w in route_weights.items():
    print(f"  {reg}:  Specialist A {w['Specialist A']:+.2f}   Specialist B {w['Specialist B']:+.2f}")

In [ ]:
#@title 📊 Visualization: the best model flips by region (double-click to view the code) { display-mode: "form" }
from IPython.display import HTML, display

def _routing_demo(spec_auc, route_weights, static_auc, dynamic_auc):
    A_COL, B_COL = '#4d8fd6', '#9a6fe2'

    def heat(a):                                   # AUC 0.5..1.0 -> red..green
        t = max(0.0, min(1.0, (a - 0.5) / 0.5))
        lo, hi = (224, 121, 109), (57, 179, 106)
        r, g, b = (round(lo[i] + (hi[i] - lo[i]) * t) for i in range(3))
        txt = '#fff' if (t < 0.30 or t > 0.62) else '#3b2d6b'
        return f'background:rgb({r},{g},{b});color:{txt}'

    rows = ''
    for name, (a0, a1) in spec_auc.items():
        rows += (f'<tr><th class="rd-rh">{name}</th>'
                 f'<td style="{heat(a0)}">{a0:.2f}</td>'
                 f'<td style="{heat(a1)}">{a1:.2f}</td></tr>')

    def sbar(a, c):                                 # AUC bar 0.5..1.0
        return f'<div class="rd-track"><div class="rd-fill" style="width:{(a-0.5)/0.5*100:.0f}%;background:{c}"></div></div>'

    # weights panel: signed weight bars around a centre line, per region
    wmax = max(abs(v) for w in route_weights.values() for v in w.values()) or 1.0
    def wbar(w, c):
        pct = abs(w) / wmax * 50
        if w >= 0:
            return (f'<div class="rd-wt"><div class="rd-half"></div>'
                    f'<div class="rd-half"><div class="rd-wpos" style="width:{pct:.0f}%;background:{c}"></div></div></div>')
        return (f'<div class="rd-wt"><div class="rd-half" style="display:flex;justify-content:flex-end">'
                f'<div class="rd-wneg" style="width:{pct:.0f}%"></div></div><div class="rd-half"></div></div>')
    wreg = ''
    for reg, w in route_weights.items():
        a, b = float(w['Specialist A']), float(w['Specialist B'])
        wreg += (f'<div class="rd-wregrow"><div class="rd-wlab">{reg}</div><div class="rd-wpair">'
                 f'<div class="rd-wrow"><span class="rd-wnm" style="color:{A_COL}">A</span>{wbar(a,A_COL)}'
                 f'<span class="rd-wv">{"+" if a>=0 else "−"}{abs(a):.2f}</span></div>'
                 f'<div class="rd-wrow"><span class="rd-wnm" style="color:{B_COL}">B</span>{wbar(b,B_COL)}'
                 f'<span class="rd-wv">{"+" if b>=0 else "−"}{abs(b):.2f}</span></div>'
                 f'</div></div>')

    lift = dynamic_auc - static_auc
    style = '''
    <style>
    .rd{font-family:system-ui,Segoe UI,Roboto,sans-serif;background:linear-gradient(135deg,#f6f8ff,#fbf5ff);
        border:1px solid #ecebff;border-radius:18px;padding:18px;max-width:800px;color:#2c2350}
    .rd-h{font-size:20px;font-weight:800;color:#3b2d6b;margin:0 0 3px}
    .rd-s{font-size:11.5px;color:#6b6685;margin:0 0 13px;line-height:1.45}
    .rd-row2{display:flex;gap:13px;flex-wrap:wrap}
    .rd-card{flex:1 1 320px;background:#fff;border:1px solid #e6e8ee;border-radius:14px;padding:13px 15px}
    .rd-wide{flex-basis:100%;margin-top:13px}
    .rd-ct{font-size:12.5px;font-weight:800;color:#2c2350;margin-bottom:9px}
    .rd-t{border-collapse:separate;border-spacing:5px;width:100%}
    .rd-t th{font-size:11px;color:#6b6685;font-weight:700;padding:3px}
    .rd-t th.rd-rh{text-align:right;color:#3b2d6b}
    .rd-t td{height:40px;text-align:center;border-radius:9px;font-size:14px;font-weight:800}
    .rd-note{font-size:10.5px;color:#a7a3bd;margin-top:7px;line-height:1.4}
    .rd-srow{display:flex;align-items:center;gap:9px;margin:9px 0}
    .rd-nm{flex:0 0 92px;font-size:12px;font-weight:700;color:#3b2d6b}
    .rd-track{flex:1;height:18px;background:#eef0f5;border-radius:8px;overflow:hidden}
    .rd-fill{height:100%;border-radius:8px;transition:width .5s ease}
    .rd-av{flex:0 0 44px;text-align:right;font-size:13px;font-weight:800;color:#3b2d6b}
    .rd-lift{display:inline-block;font-size:12px;font-weight:800;color:#1f7a45;background:#eafaf0;border-radius:7px;padding:3px 10px;margin-top:6px}
    /* weights panel */
    .rd-wregrow{display:flex;align-items:center;gap:12px;margin:8px 0}
    .rd-wlab{flex:0 0 70px;font-size:11.5px;font-weight:800;color:#3b2d6b}
    .rd-wpair{flex:1;display:flex;flex-direction:column;gap:5px}
    .rd-wrow{display:flex;align-items:center;gap:8px}
    .rd-wnm{flex:0 0 16px;font-size:12px;font-weight:800}
    .rd-wt{flex:1;display:flex;height:15px;background:#f4f4fa;border-radius:6px;overflow:hidden;border:1px solid #eceaf5}
    .rd-half{width:50%;display:flex;align-items:center}
    .rd-half:first-child{border-right:1px dashed #d8d5ea}
    .rd-wpos{height:100%;border-radius:0 4px 4px 0}
    .rd-wneg{height:100%;background:linear-gradient(90deg,#e0796d,#d4604f);border-radius:4px 0 0 4px}
    .rd-wv{flex:0 0 40px;text-align:right;font-size:11.5px;font-weight:800;color:#3b2d6b}
    .rd-cap{font-size:11.5px;color:#6b6685;line-height:1.5;margin-top:12px;background:#fff;border:1px solid #ecebff;
            border-radius:11px;padding:9px 11px}.rd-cap b{color:#3b2d6b}
    </style>'''

    html = style + f'''
    <div class="rd">
      <div class="rd-h">🧪 A case where routing wins big</div>
      <div class="rd-s">A <b>separate, rigged dataset</b>: two specialists, each expert in one region and a coin-flip
      in the other. The winner <b>flips</b> between regions &mdash; so the meta-model's job is to send each machine to
      the right expert. (These AUCs are their own story, not a comparison with the 0.911 maintenance result.)</div>
      <div class="rd-row2">
        <div class="rd-card">
          <div class="rd-ct">🎯 Each specialist's AUC, by region</div>
          <table class="rd-t">
            <tr><th></th><th>region 0</th><th>region 1</th></tr>
            {rows}
          </table>
          <div class="rd-note">Green = expert (high AUC), red = useless (≈0.5). The green cells sit on the
          <b>diagonal</b> &mdash; that is the ranking flipping.</div>
        </div>
        <div class="rd-card">
          <div class="rd-ct">🏁 Static blend vs feature-aware blend</div>
          <div class="rd-srow"><div class="rd-nm">Static</div>{sbar(static_auc, "#b8b4c8")}<div class="rd-av">{static_auc:.3f}</div></div>
          <div class="rd-srow"><div class="rd-nm">Dynamic</div>{sbar(dynamic_auc, "linear-gradient(90deg,#667eea,#764ba2)")}<div class="rd-av">{dynamic_auc:.3f}</div></div>
          <div class="rd-note">bars scaled from AUC 0.5 (useless) to 1.0 (perfect)</div>
          <div class="rd-lift">dynamic wins by +{lift:.3f} AUC</div>
        </div>
        <div class="rd-card rd-wide">
          <div class="rd-ct">⚖️ The weight the dynamic blend gives each specialist &mdash; and how it swaps with the region</div>
          {wreg}
          <div class="rd-note">This is the routing made explicit: in region 0 the blend is almost all
          <span style="color:#4d8fd6;font-weight:700">Specialist A</span>; in region 1 it flips to almost all
          <span style="color:#9a6fe2;font-weight:700">Specialist B</span>. A static stack is stuck with one row of weights for both.</div>
        </div>
      </div>
      <div class="rd-cap">A <b>static</b> weighting must pick one compromise &mdash; roughly &ldquo;half-trust each
      specialist everywhere&rdquo; &mdash; so it is mediocre in both regions ({static_auc:.3f}). The <b>dynamic</b> blend
      reads the region flag and hands each machine to its expert, getting close to perfect ({dynamic_auc:.3f}).
      <b>This</b> is when feature-aware stacking is worth it: diverse models whose strengths depend on the input.
      On the maintenance data the lift was tiny only because one model (XGBoost) was best almost everywhere.</div>
    </div>'''
    display(HTML(html))

_routing_demo(spec_auc, route_weights, static_auc, dynamic_auc)